In [ ]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import torch.nn.functional as F
import torch
import json
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data import DataLoader, Dataset
import pandas as pd

# Extract features of labels and durations. 

In [ ]:
import pandas as pd

data = pd.read_csv('labeled_data_3000-6000.csv')
df = pd.DataFrame(data)

# Encoding label names
label_encoding = {
    'Follow_behind': 0,
    'Slow_down': 1,
    'Catch_up': 2,
    'Speed_up': 3,
    'Fall_behind': 4,
    'Hold_speed': 5
}

# Define a function to calculate durations
def calculate_durations(points):
    return [int(points[i+1]) - int(points[i]) for i in range(len(points)-1)]

# Initialize lists to store encoded label sequences and duration sequences
encoded_labels = []
durations = []
for _, row in df.iterrows():
    label_sequence = []
    labels = eval(row['Label'])
    for label in labels[1:]:
        if label in label_encoding:
            label_sequence.append(label_encoding[label])
        else:
            print(f"Warning: Label '{label}' not found in label encoding dictionary.")
    
    duration_sequence = calculate_durations(eval(row['Point']))
    
    encoded_labels.append(label_sequence)
    durations.append(duration_sequence)
print(encoded_labels)
# Convert the lists to NumPy arrays
encoded_labels = np.array(encoded_labels, dtype=object)
durations = np.array(durations,dtype=object)

for i in range(min(3, len(encoded_labels))):  # Print data for the first three drivers or less if there are fewer drivers
    print(f"Driver {i+1}:")
    print("  Encoded label sequence:", encoded_labels[i])
    print("  Duration sequence:", durations[i])


In [ ]:
# Check if the label match with the duration
import pandas as pd
import numpy as np

# Load data
data = pd.read_csv('labeled_data_3000-6000.csv')
df = pd.DataFrame(data)

# Encoding label names
label_encoding = {
    'Follow_behind': 0,
    'Slow_down': 1,
    'Catch_up': 2,
    'Speed_up': 3,
    'Fall_behind': 4,
    'Hold_speed': 5
}

# Define a function to calculate durations
def calculate_durations(points):
    return [int(points[i+1]) - int(points[i]) for i in range(len(points)-1)]

# Initialize lists to store encoded label sequences and duration sequences
encoded_labels = []
durations = []
mismatched_rows = 0  # Counter for mismatched rows

for idx, row in df.iterrows():
    labels = eval(row['Label'])
    points = eval(row['Point'])

    if not isinstance(labels, list) or not isinstance(points, list):
        print(f"Invalid data format at row {idx}: Labels or Points are not lists.")
        continue

    label_sequence = [label_encoding[label] for label in labels[1:] if label in label_encoding]
    duration_sequence = calculate_durations(points)

    if len(label_sequence) != len(duration_sequence):
        print(f"Warning: Label and duration sequence lengths do not match at row {idx}")
        mismatched_rows += 1

    encoded_labels.append(label_sequence)
    durations.append(duration_sequence)

# Convert the lists to NumPy arrays
encoded_labels = np.array(encoded_labels, dtype=object)
durations = np.array(durations, dtype=object)

for i in range(min(3, len(encoded_labels))):  # Print data for the first three drivers or less if there are fewer drivers
    print(f"Driver {i+1}:")
    print("  Encoded label sequence:", encoded_labels[i])
    print("  Duration sequence:", durations[i])

# Print the number of mismatched rows
print(f"Total number of mismatched rows: {mismatched_rows}")


# Visualize probability

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
#set font
plt.rcParams['font.family'] = 'Arial'
# Define your transition probabilities
transitions = {
    "FollowBehind": {"FollowBehind": 0.3902, "CatchUp": 0.1313, "FallBehind": 0.1560, "SpeedUp": 0.1207, "SlowDown": 0.0899, "HoldSpeed": 0.1119},
    "CatchUp": {"SlowDown": 0.2578, "FallBehind": 0.1383, "HoldSpeed": 0.1060, "SpeedUp": 0.1152, "FollowBehind": 0.2572, "CatchUp": 0.1255},
    "SlowDown": {"FollowBehind": 0.2148, "FallBehind": 0.1785, "HoldSpeed": 0.0970, "SlowDown": 0.0704, "SpeedUp": 0.3119, "CatchUp": 0.1274},
    "FallBehind": {"FollowBehind": 0.2945, "HoldSpeed": 0.1137, "SpeedUp": 0.2623, "CatchUp": 0.1170, "SlowDown": 0.0644, "FallBehind": 0.1481},
    "SpeedUp": {"FallBehind": 0.1861, "SpeedUp": 0.1481, "FollowBehind": 0.2622, "SlowDown": 0.1324, "CatchUp": 0.1927, "HoldSpeed": 0.0786},
    "HoldSpeed": {"SlowDown": 0.0884, "FollowBehind": 0.2911, "FallBehind": 0.1337, "HoldSpeed": 0.2514, "CatchUp": 0.1337, "SpeedUp": 0.1017}
}

# Create a directed graph
G = nx.DiGraph()

# Add nodes and edges
for state, edges in transitions.items():
    G.add_node(state)
    for dest_state, prob in edges.items():
        G.add_edge(state, dest_state, weight=prob, label=f"{prob:.2f}")

# Configure the plot's figure size
plt.figure(figsize=(10, 7))

# Position nodes using a layout
pos = nx.spring_layout(G, seed=42)  # Seed for reproducible layout

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_color='dodgerblue', node_size=2000, alpha=0.8)

# Draw edges with varying width based on probability
edges = G.edges(data=True)
edge_colors = [data['weight'] for _, _, data in edges]

nx.draw_networkx_edges(G, pos, arrowstyle='-|>', arrowsize=20,  edge_cmap=plt.cm.cool, edge_color=edge_colors,min_target_margin=23, min_source_margin=23, connectionstyle='arc3,rad=0.1') #edge_color=edge_colors, width=[5*data['weight'] for _, _, data in edges]

# Draw labels
label_pos = {k: [v[0], v[1]] for k, v in pos.items()}  # Shift labels up
# nx.draw_networkx_labels(G, label_pos, font_size=14, font_family="Arial", font_weight='bold')

# Draw edge labels///////////""
# edge_labels = nx.get_edge_attributes(G, 'label')
# nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red')
for (u, v, d) in G.edges(data=True):
    label = f"{d['label']}"
    edge_pos = pos[u], pos[v]
    x_label_pos1 = edge_pos[0][0] * 2/3 + edge_pos[1][0] * 1/3
    y_label_pos1 = edge_pos[0][1] * 2/3 + edge_pos[1][1] * 1/3
    x_label_pos2 = edge_pos[0][0] * 1/3 + edge_pos[1][0] * 2/3
    y_label_pos2 = edge_pos[0][1] * 1/3 + edge_pos[1][1] * 2/3
    plt.text(x_label_pos1, y_label_pos1, label, color='red', fontsize=12, ha='center', va='center')
    # plt.text(x_label_pos2, y_label_pos2, label, color='blue', fontsize=12, ha='center', va='center')


# Adjust the plot margins
plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05)

# Show the graph
plt.title('State Transition Probabilities')
plt.axis('off')  # Turn off the axis
plt.show()


# Data preparation

In [ ]:
# Data preparation
import pandas as pd

data = pd.read_csv('labeled_data_3000-6000.csv')
df = pd.DataFrame(data)

# Encoding label names
label_encoding = {
    'Speed_up': 0,
    'Slow_down': 1,
    'Hold_speed': 2,
    'Catch_up': 3,
    'Fall_behind': 4,
    'Follow_behind': 5,
}

# Define a function to calculate durations
def calculate_durations(points):
    return [int(points[i+1]) - int(points[i]) for i in range(len(points)-1)]

# Initialize lists to store encoded label sequences and duration sequences
encoded_labels = []
durations = []
for _, row in df.iterrows():
    label_sequence = []
    labels = eval(row['Label'])
    for label in labels[1:]:
        if label in label_encoding:
            label_sequence.append(label_encoding[label])
        else:
            print(f"Warning: Label '{label}' not found in label encoding dictionary.")
    
    duration_sequence = calculate_durations(eval(row['Point']))
    
    encoded_labels.append(label_sequence)
    durations.append(duration_sequence)

# Convert the lists to NumPy arrays
encoded_labels = np.array(encoded_labels,dtype=object)
durations = np.array(durations,dtype=object)
durations = durations

# main model (KEP+D LSTM)

In [ ]:
#zq added
def criterion_duration20(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual.detach().numpy()
    sigma = 20 # Assuming the second value in duration_stats is variance
    theta = 80
    #tensor.detach().numpy()
    predicted = predicted.detach().numpy()
    actual = actual.detach().numpy()
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = np.exp(y0)
    
    else:
        y0 = 0
    return -np.log(y0+1e-6)

def criterion_duration2(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual
    sigma = torch.tensor(20)# Assuming the second value in duration_stats is variance
    theta = torch.tensor(80000)
    #tensor.detach().numpy()
    predicted = predicted
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = torch.exp(y0)
 
    else:
        y1 = torch.tensor(0)
    return -torch.log(y1+1e-6)

def criterion_duration(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual #49
    sigma = 31  # Assuming the second value in duration_stats is variance

    # Calculate the z-score for the predicted value
    z_score = (predicted - mu) / sigma
    abs_z_score = torch.abs(z_score)

    # Determine the loss based on the z-score
    if abs_z_score <= 1:
        # Within 1 sigma (68% of data falls here)
        return  1* abs(predicted - mu)
    elif abs_z_score <= 2:
        # Within 2 sigma (95% of data falls here)
        return 1.5 * abs(predicted - mu)
    elif abs_z_score <= 3:
        # Within 3 sigma (99.7% of data falls here)
        return abs(predicted - mu)
    else:
        # Beyond 3 sigma (outliers)
        return 2.5 * abs(predicted - mu) # Penalty for being outside 3 sigma


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
# import matplotlib.pyplot as plt

# Define transition probabilities
transitions = {
    "Follow_behind": {"Follow_behind": 0.3902, "Catch_up": 0.1313, "Fall_behind": 0.1560, "Speed_up": 0.1207, "Slow_down": 0.0899, "Hold_speed": 0.1119},
    "Catch_up": {"Slow_down": 0.2578, "Fall_behind": 0.1383, "Hold_speed": 0.1060, "Speed_up": 0.1152, "Follow_behind": 0.2572, "Catch_up": 0.1255},
    "Slow_down": {"Follow_behind": 0.2148, "Fall_behind": 0.1785, "Hold_speed": 0.0970, "Slow_down": 0.0704, "Speed_up": 0.3119, "Catch_up": 0.1274},
    "Fall_behind": {"Follow_behind": 0.2945, "Hold_speed": 0.1137, "Speed_up": 0.2623, "Catch_up": 0.1170, "Slow_down": 0.0644, "Fall_behind": 0.1481},
    "Speed_up": {"Fall_behind": 0.1861, "Speed_up": 0.1481, "Follow_behind": 0.2622, "Slow_down": 0.1324, "Catch_up": 0.1927, "Hold_speed": 0.0786},
    "Hold_speed": {"Slow_down": 0.0884, "Follow_behind": 0.2911, "Fall_behind": 0.1337, "Hold_speed": 0.2514, "Catch_up": 0.1337, "Speed_up": 0.1017}
}

# Convert transitions dictionary to tensor
transition_probs = np.zeros((6, 6))
pattern_indices = {"Speed_up": 0, "Slow_down": 1, "Hold_speed": 2, "Catch_up": 3, "Fall_behind": 4, "Follow_behind": 5}

for from_pattern, to_patterns in transitions.items():
    from_index = pattern_indices[from_pattern]
    for to_pattern, prob in to_patterns.items():
        to_index = pattern_indices[to_pattern]
        transition_probs[from_index, to_index] = prob

# Convert list data to numpy arrays
encoded_labels = np.array([np.array(x) for x in encoded_labels], dtype=object)
durations = np.array([np.array(x) for x in durations], dtype=object)

# Duration statistics from the provided data
duration_stats = {
    0: [53.776271, 1210.973094, 1.083641, 1.866133],
    1: [54.357868, 1152.487722, 0.899142, 0.934809],
    2: [47.841042, 893.930387, 1.382377, 2.465581],
    3: [47.197166, 906.236071, 1.588814, 3.821576],
    4: [51.427919, 1031.589774, 1.427946, 2.731621],
    5: [40.124668, 663.546658, 1.670533, 4.203144]
}

# Convert to numpy array for easier indexing
duration_stats = {k: np.array(v) for k, v in duration_stats.items()}

class DriverDataset(Dataset):
    def __init__(self, encoded_labels, durations, transition_probs, duration_stats, train=True):
        self.inputs = []
        self.labels = []
        self.durations = []
        self.transition_probs = transition_probs
        self.duration_stats = duration_stats
        self.train = train

        for labels, duration in zip(encoded_labels, durations):
            if train:
                if len(labels) > 1 and len(duration) > 1:
                    self.inputs.append((labels[:-1], labels[-1], duration[-1]))
            else:
                if len(labels) > 0 and len(duration) > 0:
                    self.inputs.append((labels, labels[-1], duration[-1]))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        seq_labels, next_label, next_duration = self.inputs[idx]
        transition_features = np.array([self.transition_probs[label] for label in seq_labels])
        stats = self.duration_stats[next_label]  # Get the stats for the next label
        return (torch.tensor(seq_labels, dtype=torch.long),
                torch.tensor(transition_features, dtype=torch.float),
                torch.tensor(next_label, dtype=torch.long),
                torch.tensor(next_duration, dtype=torch.float),
                torch.tensor(stats, dtype=torch.float))

# Split data into training and validation sets
train_size = int(0.8 * len(encoded_labels))
val_size = len(encoded_labels) - train_size
train_dataset = DriverDataset(encoded_labels[:train_size], durations[:train_size], transition_probs, duration_stats=duration_stats, train=True) #zq modified
val_dataset = DriverDataset(encoded_labels[train_size:], durations[train_size:], transition_probs,duration_stats=duration_stats, train=False) #zq modified

def collate_fn(batch):
    # Filter out empty sequences
    batch = [item for item in batch if len(item[0]) > 0]
    
    # Sort the batch in the descending order of sequence length
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    sequences, transitions, labels, durations, stats = zip(*batch)
    lengths = [len(seq) for seq in sequences]
    
    # Pad the sequences and transition features to make them of the same size
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=10)
    transitions_padded = pad_sequence(transitions, batch_first=True, padding_value=10)  # Padding for float tensor
    
    # Pack the sequences
    packed_sequences = pack_padded_sequence(sequences_padded, lengths, batch_first=True, enforce_sorted=False)
    packed_transitions = pack_padded_sequence(transitions_padded, lengths, batch_first=True, enforce_sorted=False)
    
    return packed_sequences, packed_transitions, torch.tensor(labels), torch.tensor(durations), torch.stack(stats), lengths

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
print(f"Number of validation samples: {len(val_dataset)}, training samples: {len(train_dataset)}")

class PatternDurationPredictor(nn.Module):
    def __init__(self, num_labels, embedding_dim=20, lstm_hidden_dim=25, num_lstm_layers=6, stats_dim=4):
        super(PatternDurationPredictor, self).__init__()
        self.embedding = nn.Embedding(num_labels, embedding_dim)
        self.lstm_input_dim = embedding_dim + num_labels  # Adjust input size
        self.lstm = nn.LSTM(input_size=self.lstm_input_dim, hidden_size=lstm_hidden_dim, num_layers=num_lstm_layers, batch_first=True, dropout=0.5)
        
        #zq added
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.sigmoid = nn.Sigmoid()
        self.attention1 = nn.Linear(lstm_hidden_dim+ stats_dim, 1)
        self.attention2 = nn.Linear(lstm_hidden_dim+ stats_dim, 1)
        self.fc1 = nn.Linear(lstm_hidden_dim + stats_dim, 10)  # For label prediction
        self.fc2 = nn.Linear(lstm_hidden_dim + stats_dim, 18)  # For duration prediction
        self.fc3 = nn.Linear(18, 10)  # For duration prediction
        self.fc4 = nn.Linear(10, 1)  # For duration prediction
        
        self.fc5 = nn.Linear(10, num_labels)  # For duration prediction
    def forward(self, x, transitions, lengths, stats):
        x = self.embedding(x)
        x = torch.cat((x, transitions), dim=2)  # Concatenate embedding and transition features

        packed_input = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)

        packed_output, (h_n, _) = self.lstm(packed_input)
        h_n = h_n[-1]  # Take the last hidden state
        # print(packed_output,'13234324')
        # Concatenate hidden state with stats
        h_n = torch.cat((h_n, stats), dim=1)

        attention_weights1 = F.softmax(self.attention1(h_n), dim=1)
        attention_output1 = (h_n * attention_weights1)

        h_n1 = self.sigmoid(attention_output1)
        # h_n = self.sigmoid(h_n)
        
        attention_weights2 = F.softmax(self.attention2(h_n1), dim=1)
        attention_output2 = (h_n1 * attention_weights2)
        h_n2 = self.sigmoid(attention_output2)
        
        label_output = self.fc5(self.fc1(h_n1))
        
        duration_output = self.fc2(h_n2)
        duration_output = self.fc3(duration_output)
        duration_output = self.fc4(duration_output)
        duration_output = self.relu(duration_output)
        
        #zq added
        label_output = self.softmax(label_output)
        
        return label_output, duration_output

# Create model
model = PatternDurationPredictor(num_labels=6)

# Training loop
num_epochs = 100
model.train()
training_losses = []

# Loss and optimizer
criterion_label = nn.CrossEntropyLoss()
criterion_duration1 = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.008)

loss_label_set = []
loss_duration_set = []
loss_label_set2 = []
loss_duration_set2 = []

for epoch in range(num_epochs):
    epoch_loss = 0
    label_epoch_loss = 0
    duration_epoch_loss = 0
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in train_loader:
        # Unpack the sequences
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
        
        # Forward pass
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_duration.squeeze(),'2e',outputs_duration.squeeze())
        
        # print(outputs_label.squeeze(),'e',labels.squeeze())
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss = loss_label + loss_duration*0.001
        loss_label_set.append(loss_label.item())
        loss_duration_set.append(loss_duration.item()*0.001)
        
        label_epoch_loss += loss_label.item()
        duration_epoch_loss += loss_duration.item()*0.001
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    label_epoch_loss /= len(train_loader)
    duration_epoch_loss /= len(train_loader)    
    loss_label_set2.append(label_epoch_loss)
    loss_duration_set2.append(duration_epoch_loss)
    epoch_loss /= len(train_loader)
    training_losses.append(epoch_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

print("Training complete")

model.eval()
correct = 0
total = 0
loss = 0
correct_duration1 = 0
correct_duration2 = 0
correct_duration3 = 0

with torch.no_grad():
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in val_loader:
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
   
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_label.squeeze(),'e',outputs_duration.squeeze())
        _, predicted_labels = torch.max(outputs_label, 1)
        total += labels.size(0)
        correct += (predicted_labels == labels).sum().item()
        
        correct_duration1 += sum((outputs_duration.squeeze() - durations)**2 <31*31) 
        correct_duration2 += sum((outputs_duration.squeeze() - durations)**2 <31*31*4) 
        correct_duration3 += sum((outputs_duration.squeeze() - durations)**2 <31*3*31*3)
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss += loss_label + loss_duration*.001
        
print(f'Accuracy of the model on the validation set: {100 * correct / total:.2f}%')
print(f'Average duration prediction error: {loss/total:.2f}')    
print(f'Accuracy of the model on the validation set: {100 * correct_duration1 / total:.2f}%, {100 * correct_duration2 / total:.2f}%, {100 * correct_duration3 / total:.2f}%')

# LSTM

In [ ]:
#zq added
def criterion_duration20(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual.detach().numpy()
    sigma = 20 # Assuming the second value in duration_stats is variance
    theta = 80
    #tensor.detach().numpy()
    predicted = predicted.detach().numpy()
    actual = actual.detach().numpy()
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = np.exp(y0)
       
    else:
        y0 = 0
    return -np.log(y0+1e-6)

def criterion_duration2(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual
    sigma = torch.tensor(20)# Assuming the second value in duration_stats is variance
    theta = torch.tensor(80000)
    #tensor.detach().numpy()
    predicted = predicted
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = torch.exp(y0)
 
    else:
        y1 = torch.tensor(0)
    return -torch.log(y1+1e-6)

def criterion_duration(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual #49
    sigma = 31  # Assuming the second value in duration_stats is variance

    # Calculate the z-score for the predicted value
    z_score = (predicted - mu) / sigma
    abs_z_score = torch.abs(z_score)

    # Determine the loss based on the z-score
    if abs_z_score <= 1:
        # Within 1 sigma (68% of data falls here)
        return  1* abs(predicted - mu)
    elif abs_z_score <= 2:
        # Within 2 sigma (95% of data falls here)
        return 1.5 * abs(predicted - mu)
    elif abs_z_score <= 3:
        # Within 3 sigma (99.7% of data falls here)
        return abs(predicted - mu)
    else:
        # Beyond 3 sigma (outliers)
        return 2.5 * abs(predicted - mu) # Penalty for being outside 3 sigma


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
# import matplotlib.pyplot as plt

# Define transition probabilities
transitions = {
    "Follow_behind": {"Follow_behind": 0.3902, "Catch_up": 0.1313, "Fall_behind": 0.1560, "Speed_up": 0.1207, "Slow_down": 0.0899, "Hold_speed": 0.1119},
    "Catch_up": {"Slow_down": 0.2578, "Fall_behind": 0.1383, "Hold_speed": 0.1060, "Speed_up": 0.1152, "Follow_behind": 0.2572, "Catch_up": 0.1255},
    "Slow_down": {"Follow_behind": 0.2148, "Fall_behind": 0.1785, "Hold_speed": 0.0970, "Slow_down": 0.0704, "Speed_up": 0.3119, "Catch_up": 0.1274},
    "Fall_behind": {"Follow_behind": 0.2945, "Hold_speed": 0.1137, "Speed_up": 0.2623, "Catch_up": 0.1170, "Slow_down": 0.0644, "Fall_behind": 0.1481},
    "Speed_up": {"Fall_behind": 0.1861, "Speed_up": 0.1481, "Follow_behind": 0.2622, "Slow_down": 0.1324, "Catch_up": 0.1927, "Hold_speed": 0.0786},
    "Hold_speed": {"Slow_down": 0.0884, "Follow_behind": 0.2911, "Fall_behind": 0.1337, "Hold_speed": 0.2514, "Catch_up": 0.1337, "Speed_up": 0.1017}
}

# Convert transitions dictionary to tensor
transition_probs = np.zeros((6, 6))
pattern_indices = {"Speed_up": 0, "Slow_down": 1, "Hold_speed": 2, "Catch_up": 3, "Fall_behind": 4, "Follow_behind": 5}

for from_pattern, to_patterns in transitions.items():
    from_index = pattern_indices[from_pattern]
    for to_pattern, prob in to_patterns.items():
        to_index = pattern_indices[to_pattern]
        transition_probs[from_index, to_index] = prob

# Convert list data to numpy arrays
encoded_labels = np.array([np.array(x) for x in encoded_labels], dtype=object)
durations = np.array([np.array(x) for x in durations_zq], dtype=object)

# Duration statistics from the provided data
duration_stats = {
    0: [53.776271, 1210.973094, 1.083641, 1.866133],
    1: [54.357868, 1152.487722, 0.899142, 0.934809],
    2: [47.841042, 893.930387, 1.382377, 2.465581],
    3: [47.197166, 906.236071, 1.588814, 3.821576],
    4: [51.427919, 1031.589774, 1.427946, 2.731621],
    5: [40.124668, 663.546658, 1.670533, 4.203144]
}

# Convert to numpy array for easier indexing
duration_stats = {k: np.array(v) for k, v in duration_stats.items()}

class DriverDataset(Dataset):
    def __init__(self, encoded_labels, durations, transition_probs, duration_stats, train=True):
        self.inputs = []
        self.labels = []
        self.durations = []
        self.transition_probs = transition_probs
        self.duration_stats = duration_stats
        self.train = train

        for labels, duration in zip(encoded_labels, durations):
            if train:
                if len(labels) > 1 and len(duration) > 1:
                    self.inputs.append((labels[:-1], labels[-1], duration[-1]))
            else:
                if len(labels) > 0 and len(duration) > 0:
                    self.inputs.append((labels, labels[-1], duration[-1]))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        seq_labels, next_label, next_duration = self.inputs[idx]
        transition_features = np.array([self.transition_probs[label] for label in seq_labels])
        stats = self.duration_stats[next_label]  # Get the stats for the next label
        return (torch.tensor(seq_labels, dtype=torch.long),
                torch.tensor(transition_features, dtype=torch.float),
                torch.tensor(next_label, dtype=torch.long),
                torch.tensor(next_duration, dtype=torch.float),
                torch.tensor(stats, dtype=torch.float))

# Split data into training and validation sets
train_size = int(0.8 * len(encoded_labels))
val_size = len(encoded_labels) - train_size
train_dataset = DriverDataset(encoded_labels[:train_size], durations[:train_size], transition_probs, duration_stats=duration_stats, train=True) #zq modified
val_dataset = DriverDataset(encoded_labels[train_size:], durations[train_size:], transition_probs,duration_stats=duration_stats, train=False) #zq modified

def collate_fn(batch):
    # Filter out empty sequences
    batch = [item for item in batch if len(item[0]) > 0]
    
    # Sort the batch in the descending order of sequence length
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    sequences, transitions, labels, durations, stats = zip(*batch)
    lengths = [len(seq) for seq in sequences]
    
    # Pad the sequences and transition features to make them of the same size
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=10)
    transitions_padded = pad_sequence(transitions, batch_first=True, padding_value=10)  # Padding for float tensor
    
    # Pack the sequences
    packed_sequences = pack_padded_sequence(sequences_padded, lengths, batch_first=True, enforce_sorted=False)
    packed_transitions = pack_padded_sequence(transitions_padded, lengths, batch_first=True, enforce_sorted=False)
    
    return packed_sequences, packed_transitions, torch.tensor(labels), torch.tensor(durations), torch.stack(stats), lengths

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

class PatternDurationPredictor(nn.Module):
    def __init__(self, num_labels, embedding_dim=20, lstm_hidden_dim=25, num_lstm_layers=6, stats_dim=4):
        super(PatternDurationPredictor, self).__init__()
        self.embedding = nn.Embedding(num_labels, embedding_dim)
        self.lstm_input_dim = embedding_dim  # Adjust input size
        self.lstm = nn.LSTM(input_size=self.lstm_input_dim, hidden_size=lstm_hidden_dim, num_layers=num_lstm_layers, batch_first=True, dropout=0.5)
        
  

        #zq added
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.sigmoid = nn.Sigmoid()
        self.attention1 = nn.Linear(lstm_hidden_dim+ stats_dim, 1)
        self.attention2 = nn.Linear(lstm_hidden_dim+ stats_dim, 1)
        self.fc1 = nn.Linear(lstm_hidden_dim, 10)  # For label prediction
        self.fc2 = nn.Linear(lstm_hidden_dim, 18)  # For duration prediction
        self.fc3 = nn.Linear(18, 10)  # For duration prediction
        self.fc4 = nn.Linear(10, 1)  # For duration prediction
        
        self.fc5 = nn.Linear(10, num_labels)  # For duration prediction
    def forward(self, x, transitions, lengths, stats):
        x = self.embedding(x)
        # x = torch.cat((x, transitions), dim=2)  # Concatenate embedding and transition features

        packed_input = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)

        packed_output, (h_n, _) = self.lstm(packed_input)
        h_n = h_n[-1]  # Take the last hidden state
        # print(packed_output,'13234324')
        # Concatenate hidden state with stats
        # h_n = torch.cat((h_n, stats), dim=1)

        # attention_weights1 = F.softmax(self.attention1(h_n), dim=1)
        # attention_output1 = (h_n * attention_weights1)

        # h_n1 = self.sigmoid(attention_output1)
        h_n = self.sigmoid(h_n)
        
        # attention_weights2 = F.softmax(self.attention2(h_n1), dim=1)
        # attention_output2 = (h_n1 * attention_weights2)
        # h_n2 = self.sigmoid(attention_output2)
        
        label_output = self.fc5(self.fc1(h_n))
        
        duration_output = self.fc2(h_n)
        duration_output = self.fc3(duration_output)
        duration_output = self.fc4(duration_output)
        duration_output = self.relu(duration_output)
        
        #zq added
        label_output = self.softmax(label_output)
        
        return label_output, duration_output

# Create model
model = PatternDurationPredictor(num_labels=6)

# Training loop
num_epochs = 100
model.train()
training_losses = []

# Loss and optimizer
criterion_label = nn.CrossEntropyLoss()
criterion_duration1 = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.008)

for epoch in range(num_epochs):
    epoch_loss = 0
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in train_loader:
        # Unpack the sequences
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
        
        # Forward pass
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_duration.squeeze(),'2e',outputs_duration.squeeze())
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss = loss_label + loss_duration*0.001
       
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    training_losses.append(epoch_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

print("Training complete")
# save training losses to csv
import pandas as pd
loss_df = pd.DataFrame({'Epoch': range(1, num_epochs + 1), 'Loss': training_losses})
loss_df.to_csv('lstm_training_losses.csv', index=False)

model.eval()
correct = 0
total = 0
loss = 0
correct_duration1 = 0
correct_duration2 = 0
correct_duration3 = 0

with torch.no_grad():
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in val_loader:
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
   
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        print(outputs_label.squeeze(),'e',outputs_duration.squeeze())
        _, predicted_labels = torch.max(outputs_label, 1)
        total += labels.size(0)
        correct += (predicted_labels == labels).sum().item()
        
        correct_duration1 += sum((outputs_duration.squeeze() - durations)**2 <31*31) 
        correct_duration2 += sum((outputs_duration.squeeze() - durations)**2 <31*31*4) 
        correct_duration3 += sum((outputs_duration.squeeze() - durations)**2 <31*3*31*3)
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss += loss_label + loss_duration*.001
        
print(f'Accuracy of the model on the validation set: {100 * correct / total:.2f}%')
print(f'Average duration prediction error: {loss/total:.2f}')    
print(f'Accuracy of the model on the validation set: {100 * correct_duration1 / total:.2f}%, {100 * correct_duration2 / total:.2f}%, {100 * correct_duration3 / total:.2f}%')

# ALSTM

In [ ]:
#zq added
def criterion_duration20(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual.detach().numpy()
    sigma = 20 # Assuming the second value in duration_stats is variance
    theta = 80
    #tensor.detach().numpy()
    predicted = predicted.detach().numpy()
    actual = actual.detach().numpy()
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = np.exp(y0)
        
    else:
        y0 = 0
    return -np.log(y0+1e-6)

def criterion_duration2(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual
    sigma = torch.tensor(20)# Assuming the second value in duration_stats is variance
    theta = torch.tensor(80000)
    #tensor.detach().numpy()
    predicted = predicted
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = torch.exp(y0)
 
    else:
        y1 = torch.tensor(0)
    return -torch.log(y1+1e-6)

def criterion_duration(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual #49
    sigma = 31  # Assuming the second value in duration_stats is variance

    # Calculate the z-score for the predicted value
    z_score = (predicted - mu) / sigma
    abs_z_score = torch.abs(z_score)

    # Determine the loss based on the z-score
    if abs_z_score <= 1:
        # Within 1 sigma (68% of data falls here)
        return  1* abs(predicted - mu)
    elif abs_z_score <= 2:
        # Within 2 sigma (95% of data falls here)
        return 1.5 * abs(predicted - mu)
    elif abs_z_score <= 3:
        # Within 3 sigma (99.7% of data falls here)
        return abs(predicted - mu)
    else:
        # Beyond 3 sigma (outliers)
        return 2.5 * abs(predicted - mu) # Penalty for being outside 3 sigma


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
# import matplotlib.pyplot as plt

# Define transition probabilities
transitions = {
    "Follow_behind": {"Follow_behind": 0.3902, "Catch_up": 0.1313, "Fall_behind": 0.1560, "Speed_up": 0.1207, "Slow_down": 0.0899, "Hold_speed": 0.1119},
    "Catch_up": {"Slow_down": 0.2578, "Fall_behind": 0.1383, "Hold_speed": 0.1060, "Speed_up": 0.1152, "Follow_behind": 0.2572, "Catch_up": 0.1255},
    "Slow_down": {"Follow_behind": 0.2148, "Fall_behind": 0.1785, "Hold_speed": 0.0970, "Slow_down": 0.0704, "Speed_up": 0.3119, "Catch_up": 0.1274},
    "Fall_behind": {"Follow_behind": 0.2945, "Hold_speed": 0.1137, "Speed_up": 0.2623, "Catch_up": 0.1170, "Slow_down": 0.0644, "Fall_behind": 0.1481},
    "Speed_up": {"Fall_behind": 0.1861, "Speed_up": 0.1481, "Follow_behind": 0.2622, "Slow_down": 0.1324, "Catch_up": 0.1927, "Hold_speed": 0.0786},
    "Hold_speed": {"Slow_down": 0.0884, "Follow_behind": 0.2911, "Fall_behind": 0.1337, "Hold_speed": 0.2514, "Catch_up": 0.1337, "Speed_up": 0.1017}
}

# Convert transitions dictionary to tensor
transition_probs = np.zeros((6, 6))
pattern_indices = {"Speed_up": 0, "Slow_down": 1, "Hold_speed": 2, "Catch_up": 3, "Fall_behind": 4, "Follow_behind": 5}

for from_pattern, to_patterns in transitions.items():
    from_index = pattern_indices[from_pattern]
    for to_pattern, prob in to_patterns.items():
        to_index = pattern_indices[to_pattern]
        transition_probs[from_index, to_index] = prob

# Convert list data to numpy arrays
encoded_labels = np.array([np.array(x) for x in encoded_labels], dtype=object)
durations = np.array([np.array(x) for x in durations_zq], dtype=object)

# Duration statistics from the provided data
duration_stats = {
    0: [53.776271, 1210.973094, 1.083641, 1.866133],
    1: [54.357868, 1152.487722, 0.899142, 0.934809],
    2: [47.841042, 893.930387, 1.382377, 2.465581],
    3: [47.197166, 906.236071, 1.588814, 3.821576],
    4: [51.427919, 1031.589774, 1.427946, 2.731621],
    5: [40.124668, 663.546658, 1.670533, 4.203144]
}

# Convert to numpy array for easier indexing
duration_stats = {k: np.array(v) for k, v in duration_stats.items()}

class DriverDataset(Dataset):
    def __init__(self, encoded_labels, durations, transition_probs, duration_stats, train=True):
        self.inputs = []
        self.labels = []
        self.durations = []
        self.transition_probs = transition_probs
        self.duration_stats = duration_stats
        self.train = train

        for labels, duration in zip(encoded_labels, durations):
            if train:
                if len(labels) > 1 and len(duration) > 1:
                    self.inputs.append((labels[:-1], labels[-1], duration[-1]))
            else:
                if len(labels) > 0 and len(duration) > 0:
                    self.inputs.append((labels, labels[-1], duration[-1]))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        seq_labels, next_label, next_duration = self.inputs[idx]
        transition_features = np.array([self.transition_probs[label] for label in seq_labels])
        stats = self.duration_stats[next_label]  # Get the stats for the next label
        return (torch.tensor(seq_labels, dtype=torch.long),
                torch.tensor(transition_features, dtype=torch.float),
                torch.tensor(next_label, dtype=torch.long),
                torch.tensor(next_duration, dtype=torch.float),
                torch.tensor(stats, dtype=torch.float))

# Split data into training and validation sets
train_size = int(0.8 * len(encoded_labels))
val_size = len(encoded_labels) - train_size
train_dataset = DriverDataset(encoded_labels[:train_size], durations[:train_size], transition_probs, duration_stats=duration_stats, train=True) #zq modified
val_dataset = DriverDataset(encoded_labels[train_size:], durations[train_size:], transition_probs,duration_stats=duration_stats, train=False) #zq modified

def collate_fn(batch):
    # Filter out empty sequences
    batch = [item for item in batch if len(item[0]) > 0]
    
    # Sort the batch in the descending order of sequence length
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    sequences, transitions, labels, durations, stats = zip(*batch)
    lengths = [len(seq) for seq in sequences]
    
    # Pad the sequences and transition features to make them of the same size
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=10)
    transitions_padded = pad_sequence(transitions, batch_first=True, padding_value=10)  # Padding for float tensor
    
    # Pack the sequences
    packed_sequences = pack_padded_sequence(sequences_padded, lengths, batch_first=True, enforce_sorted=False)
    packed_transitions = pack_padded_sequence(transitions_padded, lengths, batch_first=True, enforce_sorted=False)
    
    return packed_sequences, packed_transitions, torch.tensor(labels), torch.tensor(durations), torch.stack(stats), lengths

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

class PatternDurationPredictor(nn.Module):
    def __init__(self, num_labels, embedding_dim=20, lstm_hidden_dim=25, num_lstm_layers=6, stats_dim=4):
        super(PatternDurationPredictor, self).__init__()
        self.embedding = nn.Embedding(num_labels, embedding_dim)
        self.lstm_input_dim = embedding_dim  # Adjust input size
        self.lstm = nn.LSTM(input_size=self.lstm_input_dim, hidden_size=lstm_hidden_dim, num_layers=num_lstm_layers, batch_first=True, dropout=0.5)

        #zq added
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.sigmoid = nn.Sigmoid()
        self.attention1 = nn.Linear(lstm_hidden_dim, 1)
        self.attention2 = nn.Linear(lstm_hidden_dim, 1)
        self.fc1 = nn.Linear(lstm_hidden_dim, 10)  # For label prediction
        self.fc2 = nn.Linear(lstm_hidden_dim, 18)  # For duration prediction
        self.fc3 = nn.Linear(18, 10)  # For duration prediction
        self.fc4 = nn.Linear(10, 1)  # For duration prediction
        
        self.fc5 = nn.Linear(10, num_labels)  # For duration prediction
    def forward(self, x, transitions, lengths, stats):
        x = self.embedding(x)
        # x = torch.cat((x, transitions), dim=2)  # Concatenate embedding and transition features

        packed_input = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)

        packed_output, (h_n, _) = self.lstm(packed_input)
        h_n = h_n[-1]  # Take the last hidden state
        # print(packed_output,'13234324')
        # Concatenate hidden state with stats
        # h_n = torch.cat((h_n, stats), dim=1)

        attention_weights1 = F.softmax(self.attention1(h_n), dim=1)
        attention_output1 = (h_n * attention_weights1)

        h_n1 = self.sigmoid(attention_output1)
        # h_n = self.sigmoid(h_n)
        
        attention_weights2 = F.softmax(self.attention2(h_n), dim=1)
        attention_output2 = (h_n * attention_weights2)
        h_n2 = self.sigmoid(attention_output2)
        
        label_output = self.fc5(self.fc1(h_n1))
        
        duration_output = self.fc2(h_n2)
        duration_output = self.fc3(duration_output)
        duration_output = self.fc4(duration_output)
        duration_output = self.relu(duration_output)
        
        #zq added
        label_output = self.softmax(label_output)
        
        return label_output, duration_output

# Create model
model = PatternDurationPredictor(num_labels=6)

# Training loop
num_epochs = 100
model.train()
training_losses = []

# Loss and optimizer
criterion_label = nn.CrossEntropyLoss()
criterion_duration1 = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.008)

for epoch in range(num_epochs):
    epoch_loss = 0
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in train_loader:
        # Unpack the sequences
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
        
        # Forward pass
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_duration.squeeze(),'2e',outputs_duration.squeeze())
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss = loss_label + loss_duration*0.001
       
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    training_losses.append(epoch_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

print("Training complete")
# save training losses to csv
import pandas as pd
loss_df = pd.DataFrame({'Epoch': range(1, num_epochs + 1), 'Loss': training_losses})
loss_df.to_csv('alstm_training_losses.csv', index=False)
model.eval()
correct = 0
total = 0
loss = 0
correct_duration1 = 0
correct_duration2 = 0
correct_duration3 = 0

with torch.no_grad():
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in val_loader:
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
   
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        print(outputs_label.squeeze(),'e',outputs_duration.squeeze())
        _, predicted_labels = torch.max(outputs_label, 1)
        total += labels.size(0)
        correct += (predicted_labels == labels).sum().item()
        
        correct_duration1 += sum((outputs_duration.squeeze() - durations)**2 <31*31) 
        correct_duration2 += sum((outputs_duration.squeeze() - durations)**2 <31*31*4) 
        correct_duration3 += sum((outputs_duration.squeeze() - durations)**2 <31*3*31*3)
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss += loss_label + loss_duration*.001
        
print(f'Accuracy of the model on the validation set: {100 * correct / total:.2f}%')
print(f'Average duration prediction error: {loss/total:.2f}')    
print(f'Accuracy of the model on the validation set: {100 * correct_duration1 / total:.2f}%, {100 * correct_duration2 / total:.2f}%, {100 * correct_duration3 / total:.2f}%')

# KEDP-LSTM

In [ ]:
#zq added
def criterion_duration20(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual.detach().numpy()
    sigma = 20 # Assuming the second value in duration_stats is variance
    theta = 80
    #tensor.detach().numpy()
    predicted = predicted.detach().numpy()
    actual = actual.detach().numpy()
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = np.exp(y0)
    
    else:
        y0 = 0
    return -np.log(y0+1e-6)

def criterion_duration2(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual
    sigma = torch.tensor(20)# Assuming the second value in duration_stats is variance
    theta = torch.tensor(80000)
    #tensor.detach().numpy()
    predicted = predicted
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = torch.exp(y0)
 
    else:
        y1 = torch.tensor(0)
    return -torch.log(y1+1e-6)

def criterion_duration(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual #49
    sigma = 31  # Assuming the second value in duration_stats is variance

    # Calculate the z-score for the predicted value
    z_score = (predicted - mu) / sigma
    abs_z_score = torch.abs(z_score)

    # Determine the loss based on the z-score
    if abs_z_score <= 1:
        # Within 1 sigma (68% of data falls here)
        return  1* abs(predicted - mu)
    elif abs_z_score <= 2:
        # Within 2 sigma (95% of data falls here)
        return 1.5 * abs(predicted - mu)
    elif abs_z_score <= 3:
        # Within 3 sigma (99.7% of data falls here)
        return abs(predicted - mu)
    else:
        # Beyond 3 sigma (outliers)
        return 2.5 * abs(predicted - mu) # Penalty for being outside 3 sigma


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
# import matplotlib.pyplot as plt

# Define transition probabilities
transitions = {
    "Follow_behind": {"Follow_behind": 0.3902, "Catch_up": 0.1313, "Fall_behind": 0.1560, "Speed_up": 0.1207, "Slow_down": 0.0899, "Hold_speed": 0.1119},
    "Catch_up": {"Slow_down": 0.2578, "Fall_behind": 0.1383, "Hold_speed": 0.1060, "Speed_up": 0.1152, "Follow_behind": 0.2572, "Catch_up": 0.1255},
    "Slow_down": {"Follow_behind": 0.2148, "Fall_behind": 0.1785, "Hold_speed": 0.0970, "Slow_down": 0.0704, "Speed_up": 0.3119, "Catch_up": 0.1274},
    "Fall_behind": {"Follow_behind": 0.2945, "Hold_speed": 0.1137, "Speed_up": 0.2623, "Catch_up": 0.1170, "Slow_down": 0.0644, "Fall_behind": 0.1481},
    "Speed_up": {"Fall_behind": 0.1861, "Speed_up": 0.1481, "Follow_behind": 0.2622, "Slow_down": 0.1324, "Catch_up": 0.1927, "Hold_speed": 0.0786},
    "Hold_speed": {"Slow_down": 0.0884, "Follow_behind": 0.2911, "Fall_behind": 0.1337, "Hold_speed": 0.2514, "Catch_up": 0.1337, "Speed_up": 0.1017}
}

# Convert transitions dictionary to tensor
transition_probs = np.zeros((6, 6))
pattern_indices = {"Speed_up": 0, "Slow_down": 1, "Hold_speed": 2, "Catch_up": 3, "Fall_behind": 4, "Follow_behind": 5}

for from_pattern, to_patterns in transitions.items():
    from_index = pattern_indices[from_pattern]
    for to_pattern, prob in to_patterns.items():
        to_index = pattern_indices[to_pattern]
        transition_probs[from_index, to_index] = prob

# Convert list data to numpy arrays
encoded_labels = np.array([np.array(x) for x in encoded_labels], dtype=object)
durations = np.array([np.array(x) for x in durations_zq], dtype=object)

# Duration statistics from the provided data
duration_stats = {
    0: [53.776271, 1210.973094, 1.083641, 1.866133],
    1: [54.357868, 1152.487722, 0.899142, 0.934809],
    2: [47.841042, 893.930387, 1.382377, 2.465581],
    3: [47.197166, 906.236071, 1.588814, 3.821576],
    4: [51.427919, 1031.589774, 1.427946, 2.731621],
    5: [40.124668, 663.546658, 1.670533, 4.203144]
}

# Convert to numpy array for easier indexing
duration_stats = {k: np.array(v) for k, v in duration_stats.items()}

class DriverDataset(Dataset):
    def __init__(self, encoded_labels, durations, transition_probs, duration_stats, train=True):
        self.inputs = []
        self.labels = []
        self.durations = []
        self.transition_probs = transition_probs
        self.duration_stats = duration_stats
        self.train = train

        for labels, duration in zip(encoded_labels, durations):
            if train:
                if len(labels) > 1 and len(duration) > 1:
                    self.inputs.append((labels[:-1], labels[-1], duration[-1]))
            else:
                if len(labels) > 0 and len(duration) > 0:
                    self.inputs.append((labels, labels[-1], duration[-1]))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        seq_labels, next_label, next_duration = self.inputs[idx]
        transition_features = np.array([self.transition_probs[label] for label in seq_labels])
        stats = self.duration_stats[next_label]  # Get the stats for the next label
        return (torch.tensor(seq_labels, dtype=torch.long),
                torch.tensor(transition_features, dtype=torch.float),
                torch.tensor(next_label, dtype=torch.long),
                torch.tensor(next_duration, dtype=torch.float),
                torch.tensor(stats, dtype=torch.float))

# Split data into training and validation sets
train_size = int(0.8 * len(encoded_labels))
val_size = len(encoded_labels) - train_size
train_dataset = DriverDataset(encoded_labels[:train_size], durations[:train_size], transition_probs, duration_stats=duration_stats, train=True) #zq modified
val_dataset = DriverDataset(encoded_labels[train_size:], durations[train_size:], transition_probs,duration_stats=duration_stats, train=False) #zq modified

def collate_fn(batch):
    # Filter out empty sequences
    batch = [item for item in batch if len(item[0]) > 0]
    
    # Sort the batch in the descending order of sequence length
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    sequences, transitions, labels, durations, stats = zip(*batch)
    lengths = [len(seq) for seq in sequences]
    
    # Pad the sequences and transition features to make them of the same size
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=10)
    transitions_padded = pad_sequence(transitions, batch_first=True, padding_value=10)  # Padding for float tensor
    
    # Pack the sequences
    packed_sequences = pack_padded_sequence(sequences_padded, lengths, batch_first=True, enforce_sorted=False)
    packed_transitions = pack_padded_sequence(transitions_padded, lengths, batch_first=True, enforce_sorted=False)
    
    return packed_sequences, packed_transitions, torch.tensor(labels), torch.tensor(durations), torch.stack(stats), lengths

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

class PatternDurationPredictor(nn.Module):
    def __init__(self, num_labels, embedding_dim=20, lstm_hidden_dim=25, num_lstm_layers=6, stats_dim=4):
        super(PatternDurationPredictor, self).__init__()
        self.embedding = nn.Embedding(num_labels, embedding_dim)
        self.lstm_input_dim = embedding_dim + num_labels
        self.lstm = nn.LSTM(input_size=self.lstm_input_dim, hidden_size=lstm_hidden_dim,
                            num_layers=num_lstm_layers, batch_first=True, dropout=0.5)

        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.sigmoid = nn.Sigmoid()

        # Fully connected layers for prediction (no attention)
        self.fc1 = nn.Linear(lstm_hidden_dim + stats_dim, 10)       # Shared input
        self.fc2 = nn.Linear(10, 10)                                 # Label path
        self.fc_label = nn.Linear(10, num_labels)                   # Label prediction

        self.fc3 = nn.Linear(10, 18)                                # Duration path
        self.fc4 = nn.Linear(18, 10)
        self.fc_duration = nn.Linear(10, 1)                         # Duration prediction

    def forward(self, x, transitions, lengths, stats):
        x = self.embedding(x)
        x = torch.cat((x, transitions), dim=2)

        packed_input = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed_input)
        h_n = h_n[-1]  # Last layer's hidden state

        h_combined = torch.cat((h_n, stats), dim=1)

        x_shared = self.fc1(h_combined)
        x_shared = self.relu(x_shared)

        # Label prediction
        label_path = self.relu(self.fc2(x_shared))
        label_output = self.softmax(self.fc_label(label_path))

        # Duration prediction
        duration_path = self.relu(self.fc3(x_shared))
        duration_path = self.relu(self.fc4(duration_path))
        duration_output = self.relu(self.fc_duration(duration_path))

        return label_output, duration_output

# Create model
model = PatternDurationPredictor(num_labels=6)

# Training loop
num_epochs = 100
model.train()
training_losses = []

# Loss and optimizer
criterion_label = nn.CrossEntropyLoss()
criterion_duration1 = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.008)

loss_label_set = []
loss_duration_set = []
loss_label_set2 = []
loss_duration_set2 = []

for epoch in range(num_epochs):
    epoch_loss = 0
    label_epoch_loss = 0
    duration_epoch_loss = 0
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in train_loader:
        # Unpack the sequences
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
        
        # Forward pass
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_duration.squeeze(),'2e',outputs_duration.squeeze())
        
        # print(outputs_label.squeeze(),'e',labels.squeeze())
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss = loss_label + loss_duration*0.001
        loss_label_set.append(loss_label.item())
        loss_duration_set.append(loss_duration.item()*0.001)
        
        label_epoch_loss += loss_label.item()
        duration_epoch_loss += loss_duration.item()*0.001
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    label_epoch_loss /= len(train_loader)
    duration_epoch_loss /= len(train_loader)    
    loss_label_set2.append(label_epoch_loss)
    loss_duration_set2.append(duration_epoch_loss)
    epoch_loss /= len(train_loader)
    training_losses.append(epoch_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

import pandas as pd
loss_df = pd.DataFrame({'Epoch': range(1, num_epochs + 1), 'Loss': training_losses})
loss_df.to_csv('kedplstm_training_losses.csv', index=False)
print("Training complete")

model.eval()
correct = 0
total = 0
loss = 0
correct_duration1 = 0
correct_duration2 = 0
correct_duration3 = 0

with torch.no_grad():
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in val_loader:
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
   
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_label.squeeze(),'e',outputs_duration.squeeze())
        _, predicted_labels = torch.max(outputs_label, 1)
        total += labels.size(0)
        correct += (predicted_labels == labels).sum().item()
        
        correct_duration1 += sum((outputs_duration.squeeze() - durations)**2 <31*31) 
        correct_duration2 += sum((outputs_duration.squeeze() - durations)**2 <31*31*4) 
        correct_duration3 += sum((outputs_duration.squeeze() - durations)**2 <31*3*31*3)
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss += loss_label + loss_duration*.001
        
print(f'Accuracy of the model on the validation set: {100 * correct / total:.2f}%')
print(f'Average duration prediction error: {loss/total:.2f}')    
print(f'Accuracy of the model on the validation set: {100 * correct_duration1 / total:.2f}%, {100 * correct_duration2 / total:.2f}%, {100 * correct_duration3 / total:.2f}%')

# KEP-ALSTM

In [ ]:
#zq added
def criterion_duration20(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual.detach().numpy()
    sigma = 20 # Assuming the second value in duration_stats is variance
    theta = 80
    #tensor.detach().numpy()
    predicted = predicted.detach().numpy()
    actual = actual.detach().numpy()
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = np.exp(y0)
       
    else:
        y0 = 0
    return -np.log(y0+1e-6)

def criterion_duration2(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual
    sigma = torch.tensor(20)# Assuming the second value in duration_stats is variance
    theta = torch.tensor(80000)
    #tensor.detach().numpy()
    predicted = predicted
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = torch.exp(y0)
 
    else:
        y1 = torch.tensor(0)
    return -torch.log(y1+1e-6)

def criterion_duration(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual #49
    sigma = 31  # Assuming the second value in duration_stats is variance

    # Calculate the z-score for the predicted value
    z_score = (predicted - mu) / sigma
    abs_z_score = torch.abs(z_score)

    # Determine the loss based on the z-score
    if abs_z_score <= 1:
        # Within 1 sigma (68% of data falls here)
        return  1* abs(predicted - mu)
    elif abs_z_score <= 2:
        # Within 2 sigma (95% of data falls here)
        return 1.5 * abs(predicted - mu)
    elif abs_z_score <= 3:
        # Within 3 sigma (99.7% of data falls here)
        return abs(predicted - mu)
    else:
        # Beyond 3 sigma (outliers)
        return 2.5 * abs(predicted - mu) # Penalty for being outside 3 sigma


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
# import matplotlib.pyplot as plt

# Define transition probabilities
transitions = {
    "Follow_behind": {"Follow_behind": 0.3902, "Catch_up": 0.1313, "Fall_behind": 0.1560, "Speed_up": 0.1207, "Slow_down": 0.0899, "Hold_speed": 0.1119},
    "Catch_up": {"Slow_down": 0.2578, "Fall_behind": 0.1383, "Hold_speed": 0.1060, "Speed_up": 0.1152, "Follow_behind": 0.2572, "Catch_up": 0.1255},
    "Slow_down": {"Follow_behind": 0.2148, "Fall_behind": 0.1785, "Hold_speed": 0.0970, "Slow_down": 0.0704, "Speed_up": 0.3119, "Catch_up": 0.1274},
    "Fall_behind": {"Follow_behind": 0.2945, "Hold_speed": 0.1137, "Speed_up": 0.2623, "Catch_up": 0.1170, "Slow_down": 0.0644, "Fall_behind": 0.1481},
    "Speed_up": {"Fall_behind": 0.1861, "Speed_up": 0.1481, "Follow_behind": 0.2622, "Slow_down": 0.1324, "Catch_up": 0.1927, "Hold_speed": 0.0786},
    "Hold_speed": {"Slow_down": 0.0884, "Follow_behind": 0.2911, "Fall_behind": 0.1337, "Hold_speed": 0.2514, "Catch_up": 0.1337, "Speed_up": 0.1017}
}

# Convert transitions dictionary to tensor
transition_probs = np.zeros((6, 6))
pattern_indices = {"Speed_up": 0, "Slow_down": 1, "Hold_speed": 2, "Catch_up": 3, "Fall_behind": 4, "Follow_behind": 5}

for from_pattern, to_patterns in transitions.items():
    from_index = pattern_indices[from_pattern]
    for to_pattern, prob in to_patterns.items():
        to_index = pattern_indices[to_pattern]
        transition_probs[from_index, to_index] = prob

# Convert list data to numpy arrays
encoded_labels = np.array([np.array(x) for x in encoded_labels], dtype=object)
durations = np.array([np.array(x) for x in durations_zq], dtype=object)

# Duration statistics from the provided data
duration_stats = {
    0: [53.776271, 1210.973094, 1.083641, 1.866133],
    1: [54.357868, 1152.487722, 0.899142, 0.934809],
    2: [47.841042, 893.930387, 1.382377, 2.465581],
    3: [47.197166, 906.236071, 1.588814, 3.821576],
    4: [51.427919, 1031.589774, 1.427946, 2.731621],
    5: [40.124668, 663.546658, 1.670533, 4.203144]
}

# Convert to numpy array for easier indexing
duration_stats = {k: np.array(v) for k, v in duration_stats.items()}

class DriverDataset(Dataset):
    def __init__(self, encoded_labels, durations, transition_probs, duration_stats, train=True):
        self.inputs = []
        self.labels = []
        self.durations = []
        self.transition_probs = transition_probs
        self.duration_stats = duration_stats
        self.train = train

        for labels, duration in zip(encoded_labels, durations):
            if train:
                if len(labels) > 1 and len(duration) > 1:
                    self.inputs.append((labels[:-1], labels[-1], duration[-1]))
            else:
                if len(labels) > 0 and len(duration) > 0:
                    self.inputs.append((labels, labels[-1], duration[-1]))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        seq_labels, next_label, next_duration = self.inputs[idx]
        transition_features = np.array([self.transition_probs[label] for label in seq_labels])
        stats = self.duration_stats[next_label]  # Get the stats for the next label
        return (torch.tensor(seq_labels, dtype=torch.long),
                torch.tensor(transition_features, dtype=torch.float),
                torch.tensor(next_label, dtype=torch.long),
                torch.tensor(next_duration, dtype=torch.float),
                torch.tensor(stats, dtype=torch.float))

# Split data into training and validation sets
train_size = int(0.8 * len(encoded_labels))
val_size = len(encoded_labels) - train_size
train_dataset = DriverDataset(encoded_labels[:train_size], durations[:train_size], transition_probs, duration_stats=duration_stats, train=True) #zq modified
val_dataset = DriverDataset(encoded_labels[train_size:], durations[train_size:], transition_probs,duration_stats=duration_stats, train=False) #zq modified

def collate_fn(batch):
    # Filter out empty sequences
    batch = [item for item in batch if len(item[0]) > 0]
    
    # Sort the batch in the descending order of sequence length
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    sequences, transitions, labels, durations, stats = zip(*batch)
    lengths = [len(seq) for seq in sequences]
    
    # Pad the sequences and transition features to make them of the same size
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=10)
    transitions_padded = pad_sequence(transitions, batch_first=True, padding_value=10)  # Padding for float tensor
    
    # Pack the sequences
    packed_sequences = pack_padded_sequence(sequences_padded, lengths, batch_first=True, enforce_sorted=False)
    packed_transitions = pack_padded_sequence(transitions_padded, lengths, batch_first=True, enforce_sorted=False)
    
    return packed_sequences, packed_transitions, torch.tensor(labels), torch.tensor(durations), torch.stack(stats), lengths

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

class PatternDurationPredictor(nn.Module):
    def __init__(self, num_labels, embedding_dim=20, lstm_hidden_dim=25, num_lstm_layers=6, stats_dim=4):
        super(PatternDurationPredictor, self).__init__()
        self.embedding = nn.Embedding(num_labels, embedding_dim)
        self.lstm_input_dim = embedding_dim + num_labels  # Adjust input size
        self.lstm = nn.LSTM(input_size=self.lstm_input_dim, hidden_size=lstm_hidden_dim, num_layers=num_lstm_layers, batch_first=True, dropout=0.5)
        
  

        #zq added
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.sigmoid = nn.Sigmoid()
        self.attention1 = nn.Linear(lstm_hidden_dim, 1)
        self.attention2 = nn.Linear(lstm_hidden_dim, 1)
        self.fc1 = nn.Linear(lstm_hidden_dim, 10)  # For label prediction
        self.fc2 = nn.Linear(lstm_hidden_dim, 18)  # For duration prediction
        self.fc3 = nn.Linear(18, 10)  # For duration prediction
        self.fc4 = nn.Linear(10, 1)  # For duration prediction
        
        self.fc5 = nn.Linear(10, num_labels)  # For duration prediction
    def forward(self, x, transitions, lengths, stats):
        x = self.embedding(x)
        x = torch.cat((x, transitions), dim=2)  # Concatenate embedding and transition features

        packed_input = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)

        packed_output, (h_n, _) = self.lstm(packed_input)
        h_n = h_n[-1]  # Take the last hidden state
        # print(packed_output,'13234324')
        # Concatenate hidden state with stats
        # h_n = torch.cat((h_n, stats), dim=1)

        attention_weights1 = F.softmax(self.attention1(h_n), dim=1)
        attention_output1 = (h_n * attention_weights1)

        h_n1 = self.sigmoid(attention_output1)
        # h_n = self.sigmoid(h_n)
        
        # attention_weights2 = F.softmax(self.attention2(h_n1), dim=1)
        # attention_output2 = (h_n1 * attention_weights2)
        # h_n2 = self.sigmoid(attention_output2)
        
        label_output = self.fc5(self.fc1(h_n1))
        
        duration_output = self.fc2(h_n1)
        duration_output = self.fc3(duration_output)
        duration_output = self.fc4(duration_output)
        duration_output = self.relu(duration_output)
        
        #zq added
        label_output = self.softmax(label_output)
        
        return label_output, duration_output

# Create model
model = PatternDurationPredictor(num_labels=6)

# Training loop
num_epochs = 100
model.train()
training_losses = []

# Loss and optimizer
criterion_label = nn.CrossEntropyLoss()
criterion_duration1 = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.008)

for epoch in range(num_epochs):
    epoch_loss = 0
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in train_loader:
        # Unpack the sequences
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
        
        # Forward pass
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_duration.squeeze(),'2e',outputs_duration.squeeze())
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss = loss_label + loss_duration*0.001
       
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    training_losses.append(epoch_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

# save training losses to csv
import pandas as pd
loss_df = pd.DataFrame({'Epoch': range(1, num_epochs + 1), 'Loss': training_losses})
loss_df.to_csv('kepalstm_training_losses.csv', index=False)
print("Training complete")

model.eval()
correct = 0
total = 0
loss = 0
correct_duration1 = 0
correct_duration2 = 0
correct_duration3 = 0

with torch.no_grad():
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in val_loader:
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
   
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        print(outputs_label.squeeze(),'e',outputs_duration.squeeze())
        _, predicted_labels = torch.max(outputs_label, 1)
        total += labels.size(0)
        correct += (predicted_labels == labels).sum().item()
        
        correct_duration1 += sum((outputs_duration.squeeze() - durations)**2 <31*31) 
        correct_duration2 += sum((outputs_duration.squeeze() - durations)**2 <31*31*4) 
        correct_duration3 += sum((outputs_duration.squeeze() - durations)**2 <31*3*31*3)
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss += loss_label + loss_duration*.001
        
print(f'Accuracy of the model on the validation set: {100 * correct / total:.2f}%')
print(f'Average duration prediction error: {loss/total:.2f}')    
print(f'Accuracy of the model on the validation set: {100 * correct_duration1 / total:.2f}%, {100 * correct_duration2 / total:.2f}%, {100 * correct_duration3 / total:.2f}%')

# KED-ALSTM

In [ ]:
#zq added
def criterion_duration20(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual.detach().numpy()
    sigma = 20 # Assuming the second value in duration_stats is variance
    theta = 80
    #tensor.detach().numpy()
    predicted = predicted.detach().numpy()
    actual = actual.detach().numpy()
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = np.exp(y0)
        
    else:
        y0 = 0
    return -np.log(y0+1e-6)

def criterion_duration2(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual
    sigma = torch.tensor(20)# Assuming the second value in duration_stats is variance
    theta = torch.tensor(80000)
    #tensor.detach().numpy()
    predicted = predicted
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = torch.exp(y0)
 
    else:
        y1 = torch.tensor(0)
    return -torch.log(y1+1e-6)

def criterion_duration(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual #49
    sigma = 31  # Assuming the second value in duration_stats is variance

    # Calculate the z-score for the predicted value
    z_score = (predicted - mu) / sigma
    abs_z_score = torch.abs(z_score)

    # Determine the loss based on the z-score
    if abs_z_score <= 1:
        # Within 1 sigma (68% of data falls here)
        return  1* abs(predicted - mu)
    elif abs_z_score <= 2:
        # Within 2 sigma (95% of data falls here)
        return 1.5 * abs(predicted - mu)
    elif abs_z_score <= 3:
        # Within 3 sigma (99.7% of data falls here)
        return abs(predicted - mu)
    else:
        # Beyond 3 sigma (outliers)
        return 2.5 * abs(predicted - mu) # Penalty for being outside 3 sigma


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
# import matplotlib.pyplot as plt

# Define transition probabilities
transitions = {
    "Follow_behind": {"Follow_behind": 0.3902, "Catch_up": 0.1313, "Fall_behind": 0.1560, "Speed_up": 0.1207, "Slow_down": 0.0899, "Hold_speed": 0.1119},
    "Catch_up": {"Slow_down": 0.2578, "Fall_behind": 0.1383, "Hold_speed": 0.1060, "Speed_up": 0.1152, "Follow_behind": 0.2572, "Catch_up": 0.1255},
    "Slow_down": {"Follow_behind": 0.2148, "Fall_behind": 0.1785, "Hold_speed": 0.0970, "Slow_down": 0.0704, "Speed_up": 0.3119, "Catch_up": 0.1274},
    "Fall_behind": {"Follow_behind": 0.2945, "Hold_speed": 0.1137, "Speed_up": 0.2623, "Catch_up": 0.1170, "Slow_down": 0.0644, "Fall_behind": 0.1481},
    "Speed_up": {"Fall_behind": 0.1861, "Speed_up": 0.1481, "Follow_behind": 0.2622, "Slow_down": 0.1324, "Catch_up": 0.1927, "Hold_speed": 0.0786},
    "Hold_speed": {"Slow_down": 0.0884, "Follow_behind": 0.2911, "Fall_behind": 0.1337, "Hold_speed": 0.2514, "Catch_up": 0.1337, "Speed_up": 0.1017}
}

# Convert transitions dictionary to tensor
transition_probs = np.zeros((6, 6))
pattern_indices = {"Speed_up": 0, "Slow_down": 1, "Hold_speed": 2, "Catch_up": 3, "Fall_behind": 4, "Follow_behind": 5}

for from_pattern, to_patterns in transitions.items():
    from_index = pattern_indices[from_pattern]
    for to_pattern, prob in to_patterns.items():
        to_index = pattern_indices[to_pattern]
        transition_probs[from_index, to_index] = prob

# Convert list data to numpy arrays
encoded_labels = np.array([np.array(x) for x in encoded_labels], dtype=object)
durations = np.array([np.array(x) for x in durations_zq], dtype=object)

# Duration statistics from the provided data
duration_stats = {
    0: [53.776271, 1210.973094, 1.083641, 1.866133],
    1: [54.357868, 1152.487722, 0.899142, 0.934809],
    2: [47.841042, 893.930387, 1.382377, 2.465581],
    3: [47.197166, 906.236071, 1.588814, 3.821576],
    4: [51.427919, 1031.589774, 1.427946, 2.731621],
    5: [40.124668, 663.546658, 1.670533, 4.203144]
}

# Convert to numpy array for easier indexing
duration_stats = {k: np.array(v) for k, v in duration_stats.items()}

class DriverDataset(Dataset):
    def __init__(self, encoded_labels, durations, transition_probs, duration_stats, train=True):
        self.inputs = []
        self.labels = []
        self.durations = []
        self.transition_probs = transition_probs
        self.duration_stats = duration_stats
        self.train = train

        for labels, duration in zip(encoded_labels, durations):
            if train:
                if len(labels) > 1 and len(duration) > 1:
                    self.inputs.append((labels[:-1], labels[-1], duration[-1]))
            else:
                if len(labels) > 0 and len(duration) > 0:
                    self.inputs.append((labels, labels[-1], duration[-1]))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        seq_labels, next_label, next_duration = self.inputs[idx]
        transition_features = np.array([self.transition_probs[label] for label in seq_labels])
        stats = self.duration_stats[next_label]  # Get the stats for the next label
        return (torch.tensor(seq_labels, dtype=torch.long),
                torch.tensor(transition_features, dtype=torch.float),
                torch.tensor(next_label, dtype=torch.long),
                torch.tensor(next_duration, dtype=torch.float),
                torch.tensor(stats, dtype=torch.float))

# Split data into training and validation sets
train_size = int(0.8 * len(encoded_labels))
val_size = len(encoded_labels) - train_size
train_dataset = DriverDataset(encoded_labels[:train_size], durations[:train_size], transition_probs, duration_stats=duration_stats, train=True) #zq modified
val_dataset = DriverDataset(encoded_labels[train_size:], durations[train_size:], transition_probs,duration_stats=duration_stats, train=False) #zq modified

def collate_fn(batch):
    # Filter out empty sequences
    batch = [item for item in batch if len(item[0]) > 0]
    
    # Sort the batch in the descending order of sequence length
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    sequences, transitions, labels, durations, stats = zip(*batch)
    lengths = [len(seq) for seq in sequences]
    
    # Pad the sequences and transition features to make them of the same size
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=10)
    transitions_padded = pad_sequence(transitions, batch_first=True, padding_value=10)  # Padding for float tensor
    
    # Pack the sequences
    packed_sequences = pack_padded_sequence(sequences_padded, lengths, batch_first=True, enforce_sorted=False)
    packed_transitions = pack_padded_sequence(transitions_padded, lengths, batch_first=True, enforce_sorted=False)
    
    return packed_sequences, packed_transitions, torch.tensor(labels), torch.tensor(durations), torch.stack(stats), lengths

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

class PatternDurationPredictor(nn.Module):
    def __init__(self, num_labels, embedding_dim=20, lstm_hidden_dim=25, num_lstm_layers=6, stats_dim=4):
        super(PatternDurationPredictor, self).__init__()
        self.embedding = nn.Embedding(num_labels, embedding_dim)
        self.lstm_input_dim = embedding_dim  # Adjust input size
        self.lstm = nn.LSTM(input_size=self.lstm_input_dim, hidden_size=lstm_hidden_dim, num_layers=num_lstm_layers, batch_first=True, dropout=0.5)
        
  

        #zq added
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.sigmoid = nn.Sigmoid()
        self.attention1 = nn.Linear(lstm_hidden_dim+ stats_dim, 1)
        self.attention2 = nn.Linear(lstm_hidden_dim+ stats_dim, 1)
        self.fc1 = nn.Linear(lstm_hidden_dim + stats_dim, 10)  # For label prediction
        self.fc2 = nn.Linear(lstm_hidden_dim + stats_dim, 18)  # For duration prediction
        self.fc3 = nn.Linear(18, 10)  # For duration prediction
        self.fc4 = nn.Linear(10, 1)  # For duration prediction
        
        self.fc5 = nn.Linear(10, num_labels)  # For duration prediction
    def forward(self, x, transitions, lengths, stats):
        x = self.embedding(x)
        # x = torch.cat((x, transitions), dim=2)  # Concatenate embedding and transition features

        packed_input = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)

        packed_output, (h_n, _) = self.lstm(packed_input)
        h_n = h_n[-1]  # Take the last hidden state
        # print(packed_output,'13234324')
        # Concatenate hidden state with stats
        h_n = torch.cat((h_n, stats), dim=1)

        # attention_weights1 = F.softmax(self.attention1(h_n), dim=1)
        # attention_output1 = (h_n * attention_weights1)

        # h_n1 = self.sigmoid(attention_output1)
        h_n = self.sigmoid(h_n)
        
        attention_weights2 = F.softmax(self.attention2(h_n), dim=1)
        attention_output2 = (h_n * attention_weights2)
        h_n2 = self.sigmoid(attention_output2)
        
        label_output = self.fc5(self.fc1(h_n))
        
        duration_output = self.fc2(h_n2)
        duration_output = self.fc3(duration_output)
        duration_output = self.fc4(duration_output)
        duration_output = self.relu(duration_output)
        
        #zq added
        label_output = self.softmax(label_output)
        
        return label_output, duration_output

# Create model
model = PatternDurationPredictor(num_labels=6)

# Training loop
num_epochs = 150
model.train()
training_losses = []

# Loss and optimizer
criterion_label = nn.CrossEntropyLoss()
criterion_duration1 = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.008)

loss_label_set = []
loss_duration_set = []
loss_label_set2 = []
loss_duration_set2 = []


for epoch in range(num_epochs):
    epoch_loss = 0
    label_epoch_loss = 0
    duration_epoch_loss = 0
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in train_loader:
        # Unpack the sequences
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
        
        # Forward pass
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_duration.squeeze(),'2e',outputs_duration.squeeze())
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss = loss_label + loss_duration*0.001
        
        loss_label_set.append(loss_label.item())
        loss_duration_set.append(loss_duration.item()*0.001)
        
        label_epoch_loss += loss_label.item()
        duration_epoch_loss += loss_duration.item()*0.001
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    label_epoch_loss /= len(train_loader)
    duration_epoch_loss /= len(train_loader)
    loss_label_set2.append(np.mean(loss_label_set))
    loss_duration_set2.append(np.mean(loss_duration_set))
    epoch_loss /= len(train_loader)
    training_losses.append(epoch_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')
print("Training complete")

# save training losses to csv
import pandas as pd
loss_df = pd.DataFrame({'Epoch': range(1, num_epochs + 1), 'Loss': training_losses})
loss_df.to_csv('kedalstm_training_losses.csv', index=False)
print("Training complete")


model.eval()
correct = 0
total = 0
loss = 0
correct_duration1 = 0
correct_duration2 = 0
correct_duration3 = 0

with torch.no_grad():
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in val_loader:
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
   
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_label.squeeze(),'e',outputs_duration.squeeze())
        _, predicted_labels = torch.max(outputs_label, 1)
        total += labels.size(0)
        correct += (predicted_labels == labels).sum().item()
        
        correct_duration1 += sum((outputs_duration.squeeze() - durations)**2 <31*31) 
        correct_duration2 += sum((outputs_duration.squeeze() - durations)**2 <31*31*4) 
        correct_duration3 += sum((outputs_duration.squeeze() - durations)**2 <31*3*31*3)
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss += loss_label + loss_duration*.001
        
print(f'Accuracy of the model on the validation set: {100 * correct / total:.2f}%')
print(f'Average duration prediction error: {loss/total:.2f}')    
print(f'Accuracy of the model on the validation set: {100 * correct_duration1 / total:.2f}%, {100 * correct_duration2 / total:.2f}%, {100 * correct_duration3 / total:.2f}%')


# KEDP-Transformer

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import math
import torch.nn.functional as F

def criterion_duration_vectorized(predicted, actual, stats):
    """
    Vectorized custom loss function for duration prediction.
    """
    mu = stats[:, 0]
    sigma = torch.sqrt(stats[:, 1])
    sigma = torch.clamp(sigma, min=1e-6) # Avoid division by zero
    z_score = torch.abs((predicted.squeeze(-1) - actual) / sigma)
    error = torch.abs(predicted.squeeze(-1) - actual)

    loss = torch.where(z_score <= 1, 1.0 * error,
           torch.where(z_score <= 2, 1.5 * error,
           torch.where(z_score <= 3, 2.0 * error, 2.5 * error)))

    return loss.mean()

# Define transition probabilities
transitions = {
    "Follow_behind": {"Follow_behind": 0.3902, "Catch_up": 0.1313, "Fall_behind": 0.1560, "Speed_up": 0.1207, "Slow_down": 0.0899, "Hold_speed": 0.1119},
    "Catch_up": {"Slow_down": 0.2578, "Fall_behind": 0.1383, "Hold_speed": 0.1060, "Speed_up": 0.1152, "Follow_behind": 0.2572, "Catch_up": 0.1255},
    "Slow_down": {"Follow_behind": 0.2148, "Fall_behind": 0.1785, "Hold_speed": 0.0970, "Slow_down": 0.0704, "Speed_up": 0.3119, "Catch_up": 0.1274},
    "Fall_behind": {"Follow_behind": 0.2945, "Hold_speed": 0.1137, "Speed_up": 0.2623, "Catch_up": 0.1170, "Slow_down": 0.0644, "Fall_behind": 0.1481},
    "Speed_up": {"Fall_behind": 0.1861, "Speed_up": 0.1481, "Follow_behind": 0.2622, "Slow_down": 0.1324, "Catch_up": 0.1927, "Hold_speed": 0.0786},
    "Hold_speed": {"Slow_down": 0.0884, "Follow_behind": 0.2911, "Fall_behind": 0.1337, "Hold_speed": 0.2514, "Catch_up": 0.1337, "Speed_up": 0.1017}
}

pattern_indices = {"Speed_up": 0, "Slow_down": 1, "Hold_speed": 2, "Catch_up": 3, "Fall_behind": 4, "Follow_behind": 5}

# Duration statistics
duration_stats = {
    0: [53.776271, 1210.973094, 1.083641, 1.866133],
    1: [54.357868, 1152.487722, 0.899142, 0.934809],
    2: [47.841042, 893.930387, 1.382377, 2.465581],
    3: [47.197166, 906.236071, 1.588814, 3.821576],
    4: [51.427919, 1031.589774, 1.427946, 2.731621],
    5: [40.124668, 663.546658, 1.670533, 4.203144]
}
duration_stats = {k: np.array(v) for k, v in duration_stats.items()}

class DriverDataset(Dataset):
    def __init__(self, encoded_labels, durations, duration_stats, train=True):
        self.inputs = []
        self.duration_stats = duration_stats

        for labels, duration in zip(encoded_labels, durations):
            # For training, the input is the sequence up to the last element,
            # and the label is the last element.
            if train:
                if len(labels) > 1:
                    self.inputs.append((labels[:-1], labels[-1], duration[-1]))
            # For validation, the input is the full sequence, and the label is the last element.
            else:
                if len(labels) > 0:
                    self.inputs.append((labels, labels[-1], duration[-1]))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        seq_labels, next_label, next_duration = self.inputs[idx]
        stats = self.duration_stats[next_label]
        return (torch.tensor(seq_labels, dtype=torch.long),
                torch.tensor(next_label, dtype=torch.long),
                torch.tensor(next_duration, dtype=torch.float),
                torch.tensor(stats, dtype=torch.float))

def collate_fn_transformer(batch):
    batch = [item for item in batch if item is not None and len(item[0]) > 0]
    if not batch:
        return None, None, None, None, None

    sequences, labels, durations, stats = zip(*batch)

    # Padding value must match the `padding_idx` in the embedding layer
    padding_value = 6
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=padding_value)

    # Create the attention mask. The mask is True for padded positions.
    attention_mask = (sequences_padded == padding_value)

    return (sequences_padded,
            torch.tensor(labels, dtype=torch.long),
            torch.tensor(durations, dtype=torch.float),
            torch.stack(stats),
            attention_mask)

# Split data and create DataLoaders
# This will fail if encoded_labels is not loaded.
try:
    train_size = int(0.8 * len(encoded_labels))
    train_dataset = DriverDataset(encoded_labels[:train_size], durations_zq[:train_size], duration_stats=duration_stats, train=True)
    val_dataset = DriverDataset(encoded_labels[train_size:], durations_zq[train_size:], duration_stats=duration_stats, train=False)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn_transformer)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn_transformer)
except NameError:
    print("="*60)
    print("!!! ERROR: `encoded_labels` and `durations_zq` not defined.")
    print("!!! Please load your data in section 1 of the script.")
    print("="*60)
    exit()


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=500):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        # Transpose for positional encoding addition: [seq_len, batch_size, embedding_dim]
        x = x.transpose(0, 1)
        x = x + self.pe[:x.size(0)]
        # Transpose back
        return self.dropout(x.transpose(0, 1))

class TransformerBaseline(nn.Module):
    def __init__(self, num_labels, feature_size, num_heads, num_layers, dim_feedforward, stats_dim=4, dropout=0.1):
        super(TransformerBaseline, self).__init__()
        self.embedding = nn.Embedding(num_labels + 1, feature_size, padding_idx=6)
        self.pos_encoder = PositionalEncoding(feature_size, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feature_size,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Prediction heads
        combined_feature_dim = feature_size + stats_dim
        self.label_head = nn.Linear(feature_size, num_labels)
        self.duration_head = nn.Sequential(
            nn.Linear(combined_feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, src_seq, src_mask, stats):
        embedded_seq = self.embedding(src_seq)
        pos_encoded_seq = self.pos_encoder(embedded_seq)

        # Pass through transformer encoder
        transformer_output = self.transformer_encoder(pos_encoded_seq, src_key_padding_mask=src_mask)

        # Use the output of the last non-padded token for prediction
        # We can simply take the last element since we are predicting what comes next
        last_token_output = transformer_output[:, -1, :]

        # Predict label
        label_logits = self.label_head(last_token_output)

        # Predict duration
        duration_input = torch.cat((last_token_output, stats), dim=1)
        predicted_duration = self.duration_head(duration_input)

        return label_logits, predicted_duration


# Model Hyperparameters
num_labels = 6
feature_size = 16     # Embedding dimension
num_heads = 4          # Number of attention heads
num_layers = 4        # Number of transformer encoder layers
dim_feedforward = 128  # Dimension of the feedforward network
dropout = 0.5

# Instantiate the model
model = TransformerBaseline(
    num_labels=num_labels,
    feature_size=feature_size,
    num_heads=num_heads,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout
)

# Loss and Optimizer
criterion_label = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
num_epochs = 100
model.train()
traiing_loss = []

for epoch in range(num_epochs):
    epoch_label_loss = 0
    epoch_duration_loss = 0

    for batch in train_loader:
        if batch[0] is None: continue
        sequences, labels, durations, stats, attention_mask = batch

        # Forward pass
        outputs_label, outputs_duration = model(sequences, attention_mask, stats)

        # Calculate losses
        # loss_label = criterion_label(outputs_label, labels)
        # loss_duration = criterion_duration_vectorized(outputs_duration, durations, stats)
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))

        # Combine losses
        total_loss = loss_label * 1.0 + loss_duration * 0.001

        # Backward and optimize
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        epoch_label_loss += loss_label.item()
        epoch_duration_loss += loss_duration.item()

    avg_label_loss = epoch_label_loss / len(train_loader)
    avg_duration_loss = epoch_duration_loss / len(train_loader)
    traiing_loss.append(avg_label_loss + 0.001 * avg_duration_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Label Loss: {avg_label_loss:.4f}, Duration Loss: {avg_duration_loss:.4f}， Total Loss: {avg_label_loss + 0.001*avg_duration_loss:.4f}')

# save the loss to csv
import pandas as pd
loss_df = pd.DataFrame(traiing_loss, columns=['Loss'])
loss_df.to_csv('transformer_baseline_loss.csv', index=False)


model.eval()
total_samples = 0
correct_labels = 0
correct_duration_1_sigma = 0
correct_duration_2_sigma = 0
correct_duration_3_sigma = 0

with torch.no_grad():
    for batch in val_loader:
        if batch[0] is None: continue
        sequences, labels, durations, stats, attention_mask = batch

        outputs_label, outputs_duration = model(sequences, attention_mask, stats)

        # Label accuracy
        _, predicted_labels = torch.max(outputs_label, 1)
        total_samples += labels.size(0)
        correct_labels += (predicted_labels == labels).sum().item()

        # Duration accuracy
        std_dev = torch.sqrt(stats[:, 1])
        error_sq = (outputs_duration.squeeze(-1) - durations).pow(2)

        correct_duration_1_sigma += (error_sq < std_dev.pow(2)).sum().item()
        correct_duration_2_sigma += (error_sq < (2 * std_dev).pow(2)).sum().item()
        correct_duration_3_sigma += (error_sq < (3 * std_dev).pow(2)).sum().item()

print("\n--- Validation Results ---")
print(f'Label Accuracy: {100 * correct_labels / total_samples:.2f}%')
print(f'Duration Accuracy (within 1σ): {100 * correct_duration_1_sigma / total_samples:.2f}%')
print(f'Duration Accuracy (within 2σ): {100 * correct_duration_2_sigma / total_samples:.2f}%')
print(f'Duration Accuracy (within 3σ): {100 * correct_duration_3_sigma / total_samples:.2f}%')

# Loss balancing

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
epochs = np.arange(1, 100 + 1)  # [1, 2, ..., 100]
plt.rcParams.update({'font.family': 'Arial'})
#font 
plt.rcParams.update({'font.size': 18})

plt.figure(figsize=(8, 5))
df = pd.read_csv('loss_sensitivity_analysis1.csv')
num_epochs = 100
plt.plot(np.arange(1, num_epochs + 1), df['label_loss'], 
         label='Pattern label loss ($\mathrm{\lambda_1}\mathrm{L}_{\mathrm{pattern}}$)', 
         color='salmon', linewidth=4, linestyle='--')  # 

plt.plot(np.arange(1, num_epochs + 1), df['duration_loss'], 
         label='Pattern duration loss ($\mathrm{\lambda_2}\mathrm{L}_{\mathrm{duration}}$)', 
         color="#79A9D3", linewidth=4)  # 

# plt.title(r'Training Losses ($\mathrm{KE_{P+D}-ALSTM}$)')
plt.xlabel('Epochs')
plt.xlim(1, 100)
# plt.xticks(np.arange(1, 101, 10))

plt.ylabel('Loss')
plt.legend()
# plt.grid(True)
plt.savefig('./loss_record/loss_balance.pdf', bbox_inches='tight', dpi=300)
# plt.show()


# Tuning lambda

In [ ]:
#zq added
def criterion_duration20(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual.detach().numpy()
    sigma = 20 # Assuming the second value in duration_stats is variance
    theta = 80
    #tensor.detach().numpy()
    predicted = predicted.detach().numpy()
    actual = actual.detach().numpy()
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = np.exp(y0)
    
    else:
        y0 = 0
    return -np.log(y0+1e-6)

def criterion_duration2(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual
    sigma = torch.tensor(20)# Assuming the second value in duration_stats is variance
    theta = torch.tensor(80000)
    #tensor.detach().numpy()
    predicted = predicted
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = torch.exp(y0)
 
    else:
        y1 = torch.tensor(0)
    return -torch.log(y1+1e-6)

def criterion_duration(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual #49
    sigma = 31  # Assuming the second value in duration_stats is variance

    # Calculate the z-score for the predicted value
    z_score = (predicted - mu) / sigma
    abs_z_score = torch.abs(z_score)

    # Determine the loss based on the z-score
    if abs_z_score <= 1:
        # Within 1 sigma (68% of data falls here)
        return  1* abs(predicted - mu)
    elif abs_z_score <= 2:
        # Within 2 sigma (95% of data falls here)
        return 1.5 * abs(predicted - mu)
    elif abs_z_score <= 3:
        # Within 3 sigma (99.7% of data falls here)
        return abs(predicted - mu)
    else:
        # Beyond 3 sigma (outliers)
        return 2.5 * abs(predicted - mu) # Penalty for being outside 3 sigma


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
# import matplotlib.pyplot as plt

# Define transition probabilities
transitions = {
    "Follow_behind": {"Follow_behind": 0.3902, "Catch_up": 0.1313, "Fall_behind": 0.1560, "Speed_up": 0.1207, "Slow_down": 0.0899, "Hold_speed": 0.1119},
    "Catch_up": {"Slow_down": 0.2578, "Fall_behind": 0.1383, "Hold_speed": 0.1060, "Speed_up": 0.1152, "Follow_behind": 0.2572, "Catch_up": 0.1255},
    "Slow_down": {"Follow_behind": 0.2148, "Fall_behind": 0.1785, "Hold_speed": 0.0970, "Slow_down": 0.0704, "Speed_up": 0.3119, "Catch_up": 0.1274},
    "Fall_behind": {"Follow_behind": 0.2945, "Hold_speed": 0.1137, "Speed_up": 0.2623, "Catch_up": 0.1170, "Slow_down": 0.0644, "Fall_behind": 0.1481},
    "Speed_up": {"Fall_behind": 0.1861, "Speed_up": 0.1481, "Follow_behind": 0.2622, "Slow_down": 0.1324, "Catch_up": 0.1927, "Hold_speed": 0.0786},
    "Hold_speed": {"Slow_down": 0.0884, "Follow_behind": 0.2911, "Fall_behind": 0.1337, "Hold_speed": 0.2514, "Catch_up": 0.1337, "Speed_up": 0.1017}
}

# Convert transitions dictionary to tensor
transition_probs = np.zeros((6, 6))
pattern_indices = {"Speed_up": 0, "Slow_down": 1, "Hold_speed": 2, "Catch_up": 3, "Fall_behind": 4, "Follow_behind": 5}

for from_pattern, to_patterns in transitions.items():
    from_index = pattern_indices[from_pattern]
    for to_pattern, prob in to_patterns.items():
        to_index = pattern_indices[to_pattern]
        transition_probs[from_index, to_index] = prob

# Convert list data to numpy arrays
encoded_labels = np.array([np.array(x) for x in encoded_labels], dtype=object)
durations = np.array([np.array(x) for x in durations_zq], dtype=object)

# Duration statistics from the provided data
duration_stats = {
    0: [53.776271, 1210.973094, 1.083641, 1.866133],
    1: [54.357868, 1152.487722, 0.899142, 0.934809],
    2: [47.841042, 893.930387, 1.382377, 2.465581],
    3: [47.197166, 906.236071, 1.588814, 3.821576],
    4: [51.427919, 1031.589774, 1.427946, 2.731621],
    5: [40.124668, 663.546658, 1.670533, 4.203144]
}

# Convert to numpy array for easier indexing
duration_stats = {k: np.array(v) for k, v in duration_stats.items()}

class DriverDataset(Dataset):
    def __init__(self, encoded_labels, durations, transition_probs, duration_stats, train=True):
        self.inputs = []
        self.labels = []
        self.durations = []
        self.transition_probs = transition_probs
        self.duration_stats = duration_stats
        self.train = train

        for labels, duration in zip(encoded_labels, durations):
            if train:
                if len(labels) > 1 and len(duration) > 1:
                    self.inputs.append((labels[:-1], labels[-1], duration[-1]))
            else:
                if len(labels) > 0 and len(duration) > 0:
                    self.inputs.append((labels, labels[-1], duration[-1]))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        seq_labels, next_label, next_duration = self.inputs[idx]
        transition_features = np.array([self.transition_probs[label] for label in seq_labels])
        stats = self.duration_stats[next_label]  # Get the stats for the next label
        return (torch.tensor(seq_labels, dtype=torch.long),
                torch.tensor(transition_features, dtype=torch.float),
                torch.tensor(next_label, dtype=torch.long),
                torch.tensor(next_duration, dtype=torch.float),
                torch.tensor(stats, dtype=torch.float))

# Split data into training and validation sets
train_size = int(0.8 * len(encoded_labels))
val_size = len(encoded_labels) - train_size
train_dataset = DriverDataset(encoded_labels[:train_size], durations[:train_size], transition_probs, duration_stats=duration_stats, train=True) #zq modified
val_dataset = DriverDataset(encoded_labels[train_size:], durations[train_size:], transition_probs,duration_stats=duration_stats, train=False) #zq modified

def collate_fn(batch):
    # Filter out empty sequences
    batch = [item for item in batch if len(item[0]) > 0]
    
    # Sort the batch in the descending order of sequence length
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    sequences, transitions, labels, durations, stats = zip(*batch)
    lengths = [len(seq) for seq in sequences]
    
    # Pad the sequences and transition features to make them of the same size
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=10)
    transitions_padded = pad_sequence(transitions, batch_first=True, padding_value=10)  # Padding for float tensor
    
    # Pack the sequences
    packed_sequences = pack_padded_sequence(sequences_padded, lengths, batch_first=True, enforce_sorted=False)
    packed_transitions = pack_padded_sequence(transitions_padded, lengths, batch_first=True, enforce_sorted=False)
    
    return packed_sequences, packed_transitions, torch.tensor(labels), torch.tensor(durations), torch.stack(stats), lengths

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

class PatternDurationPredictor(nn.Module):
    def __init__(self, num_labels, embedding_dim=20, lstm_hidden_dim=25, num_lstm_layers=6, stats_dim=4):
        super(PatternDurationPredictor, self).__init__()
        self.embedding = nn.Embedding(num_labels, embedding_dim)
        self.lstm_input_dim = embedding_dim + num_labels  # Adjust input size
        self.lstm = nn.LSTM(input_size=self.lstm_input_dim, hidden_size=lstm_hidden_dim, num_layers=num_lstm_layers, batch_first=True, dropout=0.5)
        
  

        #zq added
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.sigmoid = nn.Sigmoid()
        self.attention1 = nn.Linear(lstm_hidden_dim+ stats_dim, 1)
        self.attention2 = nn.Linear(lstm_hidden_dim+ stats_dim, 1)
        self.fc1 = nn.Linear(lstm_hidden_dim + stats_dim, 10)  # For label prediction
        self.fc2 = nn.Linear(lstm_hidden_dim + stats_dim, 18)  # For duration prediction
        self.fc3 = nn.Linear(18, 10)  # For duration prediction
        self.fc4 = nn.Linear(10, 1)  # For duration prediction
        
        self.fc5 = nn.Linear(10, num_labels)  # For duration prediction
    def forward(self, x, transitions, lengths, stats):
        x = self.embedding(x)
        x = torch.cat((x, transitions), dim=2)  # Concatenate embedding and transition features

        packed_input = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)

        packed_output, (h_n, _) = self.lstm(packed_input)
        h_n = h_n[-1]  # Take the last hidden state
        # print(packed_output,'13234324')
        # Concatenate hidden state with stats
        h_n = torch.cat((h_n, stats), dim=1)

        attention_weights1 = F.softmax(self.attention1(h_n), dim=1)
        attention_output1 = (h_n * attention_weights1)

        h_n1 = self.sigmoid(attention_output1)
        # h_n = self.sigmoid(h_n)
        
        attention_weights2 = F.softmax(self.attention2(h_n1), dim=1)
        attention_output2 = (h_n1 * attention_weights2)
        h_n2 = self.sigmoid(attention_output2)
        
        label_output = self.fc5(self.fc1(h_n1))
        
        duration_output = self.fc2(h_n2)
        duration_output = self.fc3(duration_output)
        duration_output = self.fc4(duration_output)
        duration_output = self.relu(duration_output)
        
        #zq added
        label_output = self.softmax(label_output)
        
        return label_output, duration_output

# Create model
model = PatternDurationPredictor(num_labels=6)

# Training loop
num_epochs = 100
model.train()
training_losses = []

# Loss and optimizer
criterion_label = nn.CrossEntropyLoss()
criterion_duration1 = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.008)

loss_label_set = []
loss_duration_set = []
loss_label_set2 = []
loss_duration_set2 = []

lambda1 = 10
lambda2 = 1

for epoch in range(num_epochs):
    epoch_loss = 0
    label_epoch_loss = 0
    duration_epoch_loss = 0
    label_epoch_loss_wo = 0
    duration_epoch_loss_wo = 0
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in train_loader:
        # Unpack the sequences
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
        
        # Forward pass
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_duration.squeeze(),'2e',outputs_duration.squeeze())
        
        # print(outputs_label.squeeze(),'e',labels.squeeze())
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss = loss_label*lambda1 + loss_duration*lambda2
        loss_label_set.append(loss_label.item()*lambda1)
        loss_duration_set.append(loss_duration.item()*lambda2)
        
        label_epoch_loss += loss_label.item()*lambda1
        duration_epoch_loss += loss_duration.item()*lambda2
        label_epoch_loss_wo += loss_label.item()
        duration_epoch_loss_wo += loss_duration.item()
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    label_epoch_loss /= len(train_loader)
    duration_epoch_loss /= len(train_loader)    
    label_epoch_loss_wo /= len(train_loader)
    duration_epoch_loss_wo /= len(train_loader)
    loss_label_set2.append(label_epoch_loss_wo)
    loss_duration_set2.append(duration_epoch_loss_wo)
    epoch_loss /= len(train_loader)
    training_losses.append(epoch_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

print("Training complete")

model.eval()
correct = 0
total = 0
loss = 0
correct_duration1 = 0
correct_duration2 = 0
correct_duration3 = 0

with torch.no_grad():
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in val_loader:
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
   
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_label.squeeze(),'e',outputs_duration.squeeze())
        _, predicted_labels = torch.max(outputs_label, 1)
        total += labels.size(0)
        correct += (predicted_labels == labels).sum().item()
        
        correct_duration1 += sum((outputs_duration.squeeze() - durations)**2 <31*31) 
        correct_duration2 += sum((outputs_duration.squeeze() - durations)**2 <31*31*4) 
        correct_duration3 += sum((outputs_duration.squeeze() - durations)**2 <31*3*31*3)
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss += loss_label*lambda1 + loss_duration*lambda2
        
print(f'Accuracy of the model on the validation set: {100 * correct / total:.2f}%')
print(f'Average duration prediction error: {loss/total:.2f}')    
print(f'Accuracy of the model on the validation set: {100 * correct_duration1 / total:.2f}%, {100 * correct_duration2 / total:.2f}%, {100 * correct_duration3 / total:.2f}%')

# Sensitivity analysis: lambda

In [ ]:
# lambda1 = 1, lambda2 = 0.001 model 1
loss_1 = training_losses
# save to a csv file
import pandas as pd
df = pd.DataFrame({
    'model': 'version2',
    'lambda1': lambda1,
    'lambda2': lambda2,
    'Epoch': np.arange(1, num_epochs + 1),
    'Loss': loss_1,
    'label_loss': loss_label_set2,
    'duration_loss': loss_duration_set2
})

df.to_csv('./loss_record/loss_sensitivity_analysis_v2_6_2.csv', index=False)


# Sensitivity analysis: hidden size and layers number

In [ ]:
#zq added
def criterion_duration20(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual.detach().numpy()
    sigma = 20 # Assuming the second value in duration_stats is variance
    theta = 80
    #tensor.detach().numpy()
    predicted = predicted.detach().numpy()
    actual = actual.detach().numpy()
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = np.exp(y0)
    
    else:
        y0 = 0
    return -np.log(y0+1e-6)

def criterion_duration2(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual
    sigma = torch.tensor(20)# Assuming the second value in duration_stats is variance
    theta = torch.tensor(80000)
    #tensor.detach().numpy()
    predicted = predicted
    if abs(predicted - actual) < theta:
        y0 = -(predicted - mu)**2/(2*sigma**2)
        y1 = torch.exp(y0)
 
    else:
        y1 = torch.tensor(0)
    return -torch.log(y1+1e-6)

def criterion_duration(predicted, actual):
    # Extract mean (mu) and standard deviation (sigma) for the actual label duration
    mu = actual #49
    sigma = 31  # Assuming the second value in duration_stats is variance

    # Calculate the z-score for the predicted value
    z_score = (predicted - mu) / sigma
    abs_z_score = torch.abs(z_score)

    # Determine the loss based on the z-score
    if abs_z_score <= 1:
        # Within 1 sigma (68% of data falls here)
        return  1* abs(predicted - mu)
    elif abs_z_score <= 2:
        # Within 2 sigma (95% of data falls here)
        return 1.5 * abs(predicted - mu)
    elif abs_z_score <= 3:
        # Within 3 sigma (99.7% of data falls here)
        return abs(predicted - mu)
    else:
        # Beyond 3 sigma (outliers)
        return 2.5 * abs(predicted - mu) # Penalty for being outside 3 sigma


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
# import matplotlib.pyplot as plt

# Define transition probabilities
transitions = {
    "Follow_behind": {"Follow_behind": 0.3902, "Catch_up": 0.1313, "Fall_behind": 0.1560, "Speed_up": 0.1207, "Slow_down": 0.0899, "Hold_speed": 0.1119},
    "Catch_up": {"Slow_down": 0.2578, "Fall_behind": 0.1383, "Hold_speed": 0.1060, "Speed_up": 0.1152, "Follow_behind": 0.2572, "Catch_up": 0.1255},
    "Slow_down": {"Follow_behind": 0.2148, "Fall_behind": 0.1785, "Hold_speed": 0.0970, "Slow_down": 0.0704, "Speed_up": 0.3119, "Catch_up": 0.1274},
    "Fall_behind": {"Follow_behind": 0.2945, "Hold_speed": 0.1137, "Speed_up": 0.2623, "Catch_up": 0.1170, "Slow_down": 0.0644, "Fall_behind": 0.1481},
    "Speed_up": {"Fall_behind": 0.1861, "Speed_up": 0.1481, "Follow_behind": 0.2622, "Slow_down": 0.1324, "Catch_up": 0.1927, "Hold_speed": 0.0786},
    "Hold_speed": {"Slow_down": 0.0884, "Follow_behind": 0.2911, "Fall_behind": 0.1337, "Hold_speed": 0.2514, "Catch_up": 0.1337, "Speed_up": 0.1017}
}

# Convert transitions dictionary to tensor
transition_probs = np.zeros((6, 6))
pattern_indices = {"Speed_up": 0, "Slow_down": 1, "Hold_speed": 2, "Catch_up": 3, "Fall_behind": 4, "Follow_behind": 5}

for from_pattern, to_patterns in transitions.items():
    from_index = pattern_indices[from_pattern]
    for to_pattern, prob in to_patterns.items():
        to_index = pattern_indices[to_pattern]
        transition_probs[from_index, to_index] = prob

# Convert list data to numpy arrays
encoded_labels = np.array([np.array(x) for x in encoded_labels], dtype=object)
durations = np.array([np.array(x) for x in durations_zq], dtype=object)

# Duration statistics from the provided data
duration_stats = {
    0: [53.776271, 1210.973094, 1.083641, 1.866133],
    1: [54.357868, 1152.487722, 0.899142, 0.934809],
    2: [47.841042, 893.930387, 1.382377, 2.465581],
    3: [47.197166, 906.236071, 1.588814, 3.821576],
    4: [51.427919, 1031.589774, 1.427946, 2.731621],
    5: [40.124668, 663.546658, 1.670533, 4.203144]
}

# Convert to numpy array for easier indexing
duration_stats = {k: np.array(v) for k, v in duration_stats.items()}

class DriverDataset(Dataset):
    def __init__(self, encoded_labels, durations, transition_probs, duration_stats, train=True):
        self.inputs = []
        self.labels = []
        self.durations = []
        self.transition_probs = transition_probs
        self.duration_stats = duration_stats
        self.train = train

        for labels, duration in zip(encoded_labels, durations):
            if train:
                if len(labels) > 1 and len(duration) > 1:
                    self.inputs.append((labels[:-1], labels[-1], duration[-1]))
            else:
                if len(labels) > 0 and len(duration) > 0:
                    self.inputs.append((labels, labels[-1], duration[-1]))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        seq_labels, next_label, next_duration = self.inputs[idx]
        transition_features = np.array([self.transition_probs[label] for label in seq_labels])
        stats = self.duration_stats[next_label]  # Get the stats for the next label
        return (torch.tensor(seq_labels, dtype=torch.long),
                torch.tensor(transition_features, dtype=torch.float),
                torch.tensor(next_label, dtype=torch.long),
                torch.tensor(next_duration, dtype=torch.float),
                torch.tensor(stats, dtype=torch.float))

# Split data into training and validation sets
train_size = int(0.8 * len(encoded_labels))
val_size = len(encoded_labels) - train_size
train_dataset = DriverDataset(encoded_labels[:train_size], durations[:train_size], transition_probs, duration_stats=duration_stats, train=True) #zq modified
val_dataset = DriverDataset(encoded_labels[train_size:], durations[train_size:], transition_probs,duration_stats=duration_stats, train=False) #zq modified

def collate_fn(batch):
    # Filter out empty sequences
    batch = [item for item in batch if len(item[0]) > 0]
    
    # Sort the batch in the descending order of sequence length
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    sequences, transitions, labels, durations, stats = zip(*batch)
    lengths = [len(seq) for seq in sequences]
    
    # Pad the sequences and transition features to make them of the same size
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=10)
    transitions_padded = pad_sequence(transitions, batch_first=True, padding_value=10)  # Padding for float tensor
    
    # Pack the sequences
    packed_sequences = pack_padded_sequence(sequences_padded, lengths, batch_first=True, enforce_sorted=False)
    packed_transitions = pack_padded_sequence(transitions_padded, lengths, batch_first=True, enforce_sorted=False)
    
    return packed_sequences, packed_transitions, torch.tensor(labels), torch.tensor(durations), torch.stack(stats), lengths

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

class PatternDurationPredictor(nn.Module):
    def __init__(self, num_labels, embedding_dim=20, lstm_hidden_dim=16, num_lstm_layers=6, stats_dim=4):
        super(PatternDurationPredictor, self).__init__()
        self.embedding = nn.Embedding(num_labels, embedding_dim)
        self.lstm_input_dim = embedding_dim + num_labels  # Adjust input size
        self.lstm = nn.LSTM(input_size=self.lstm_input_dim, hidden_size=lstm_hidden_dim, num_layers=num_lstm_layers, batch_first=True, dropout=0.5)
        
  

        #zq added
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.sigmoid = nn.Sigmoid()
        self.attention1 = nn.Linear(lstm_hidden_dim+ stats_dim, 1)
        self.attention2 = nn.Linear(lstm_hidden_dim+ stats_dim, 1)
        self.fc1 = nn.Linear(lstm_hidden_dim + stats_dim, 10)  # For label prediction
        self.fc2 = nn.Linear(lstm_hidden_dim + stats_dim, 18)  # For duration prediction
        self.fc3 = nn.Linear(18, 10)  # For duration prediction
        self.fc4 = nn.Linear(10, 1)  # For duration prediction
        
        self.fc5 = nn.Linear(10, num_labels)  # For duration prediction
    def forward(self, x, transitions, lengths, stats):
        x = self.embedding(x)
        x = torch.cat((x, transitions), dim=2)  # Concatenate embedding and transition features

        packed_input = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)

        packed_output, (h_n, _) = self.lstm(packed_input)
        h_n = h_n[-1]  # Take the last hidden state
        # print(packed_output,'13234324')
        # Concatenate hidden state with stats
        h_n = torch.cat((h_n, stats), dim=1)

        attention_weights1 = F.softmax(self.attention1(h_n), dim=1)
        attention_output1 = (h_n * attention_weights1)

        h_n1 = self.sigmoid(attention_output1)
        # h_n = self.sigmoid(h_n)
        
        attention_weights2 = F.softmax(self.attention2(h_n1), dim=1)
        attention_output2 = (h_n1 * attention_weights2)
        h_n2 = self.sigmoid(attention_output2)
        
        label_output = self.fc5(self.fc1(h_n1))
        
        duration_output = self.fc2(h_n2)
        duration_output = self.fc3(duration_output)
        duration_output = self.fc4(duration_output)
        duration_output = self.relu(duration_output)
        
        #zq added
        label_output = self.softmax(label_output)
        
        return label_output, duration_output

# Create model
model = PatternDurationPredictor(num_labels=6)

# Training loop
num_epochs = 100
model.train()
training_losses = []

# Loss and optimizer
criterion_label = nn.CrossEntropyLoss()
criterion_duration1 = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.008)

loss_label_set = []
loss_duration_set = []
loss_label_set2 = []
loss_duration_set2 = []

for epoch in range(num_epochs):
    epoch_loss = 0
    label_epoch_loss = 0
    duration_epoch_loss = 0
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in train_loader:
        # Unpack the sequences
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
        
        # Forward pass
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_duration.squeeze(),'2e',outputs_duration.squeeze())
        
        # print(outputs_label.squeeze(),'e',labels.squeeze())
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss = loss_label*1 + loss_duration*.001
        loss_label_set.append(loss_label.item()*1)
        loss_duration_set.append(loss_duration.item()*.001)
        
        label_epoch_loss += loss_label.item()*1
        duration_epoch_loss += loss_duration.item()*.001
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    label_epoch_loss /= len(train_loader)
    duration_epoch_loss /= len(train_loader)    
    loss_label_set2.append(label_epoch_loss)
    loss_duration_set2.append(duration_epoch_loss)
    epoch_loss /= len(train_loader)
    training_losses.append(epoch_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

print("Training complete")

model.eval()
correct = 0
total = 0
loss = 0
correct_duration1 = 0
correct_duration2 = 0
correct_duration3 = 0

with torch.no_grad():
    for packed_sequences, packed_transitions, labels, durations, stats, lengths in val_loader:
        inputs, _ = pad_packed_sequence(packed_sequences, batch_first=True)
        transitions, _ = pad_packed_sequence(packed_transitions, batch_first=True)
   
        outputs_label, outputs_duration = model(inputs, transitions, lengths, stats)
        # print(outputs_label.squeeze(),'e',outputs_duration.squeeze())
        _, predicted_labels = torch.max(outputs_label, 1)
        total += labels.size(0)
        correct += (predicted_labels == labels).sum().item()
        
        correct_duration1 += sum((outputs_duration.squeeze() - durations)**2 <31*31) 
        correct_duration2 += sum((outputs_duration.squeeze() - durations)**2 <31*31*4) 
        correct_duration3 += sum((outputs_duration.squeeze() - durations)**2 <31*3*31*3)
        loss_label = criterion_label(outputs_label, labels)
        loss_duration = sum(criterion_duration(pred, actual) 
                            for pred, actual in zip(outputs_duration.squeeze(), durations))
        loss += loss_label*1 + loss_duration*.001
        
print(f'Accuracy of the model on the validation set: {100 * correct / total:.2f}%')
print(f'Average duration prediction error: {loss/total:.2f}')    
print(f'Accuracy of the model on the validation set: {100 * correct_duration1 / total:.2f}%, {100 * correct_duration2 / total:.2f}%, {100 * correct_duration3 / total:.2f}%')

# Training losses

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Load the loss data from CSV files
lstm_loss = pd.read_csv('lstm_training_losses.csv')['Loss']
a_lstm = pd.read_csv('alstm_training_losses.csv')['Loss']
lstm_p_loss = pd.read_csv('kepalstm_training_losses.csv')['Loss']
lstm_d_loss = pd.read_csv('kedalstm_training_losses.csv')['Loss']
transformer_loss = pd.read_csv('transformer_baseline_loss.csv')['Loss']
mainloss_loss = pd.read_csv('loss_sensitivity_analysis1.csv')['Loss']
pd_lstm = pd.read_csv('kedplstm_training_losses.csv')['Loss']


# lstm_loss = np.loadtxt('lstm_training_losses.csv', delimiter=',')
# a_lstm = np.loadtxt('alstm_training_losses.csv', delimiter=',')
# lstm_p_loss = np.loadtxt('kepalsm_training_losses.csv', delimiter=',')
# lstm_d_loss = np.loadtxt('kedalsm_training_losses.csv', delimiter=',')
# transformer_loss = np.loadtxt('transformer_baseline_loss.csv', delimiter=',')
# mainloss_loss = np.loadtxt('loss_sensitivity_analysis1.csv', delimiter=',')

x_axis = np.arange(100)
plt.figure(figsize=(10, 6))
plt.rcParams.update({'font.family': 'Arial'})
#font 
plt.rcParams.update({'font.size': 16})
#using different colors and styles for each line
linestyle = ['dashdot', '--', 'dashed', '-', '--', '-.', ':']
plt.plot(x_axis,lstm_loss, label='LSTM',color = 'salmon', linestyle= linestyle[0], linewidth=2)
plt.plot(x_axis,a_lstm, label='ALSTM',color='royalblue', linestyle= linestyle[1], linewidth=2,marker='o', markersize=3)
plt.plot(x_axis, lstm_p_loss, label=r'$\mathrm{KE}_{\mathit{P}}$-ALSTM', color='springgreen', linestyle=linestyle[2], linewidth=2)
plt.plot(x_axis, lstm_d_loss, label=r'$\mathrm{KE}_{\mathit{D}}$-ALSTM', color='darkorange', linestyle=linestyle[3], linewidth=2)
plt.plot(x_axis, pd_lstm, label=r'$\mathrm{KE}_{\mathit{P+D}}$-LSTM', color='tomato', marker= '+',linestyle=linestyle[3], linewidth=2)
plt.plot(x_axis, mainloss_loss, label=r'$\mathrm{KE}_{\mathit{P+D}}$-ALSTM', color='blueviolet', linestyle=linestyle[4], linewidth=2,marker='x', markersize=3)
plt.plot(x_axis,transformer_loss, label=r'$\mathrm{KE}_{\mathit{P+D}}$-Transformer',color='darkred', linestyle= linestyle[5], linewidth=2)
plt.ylim(1.9, 3.6)
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend()
#save pdf
plt.savefig('loss.pdf', bbox_inches='tight',dpi=300)
plt.show()

Training losses: label

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#fontsize
plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'font.family': 'Arial'})
#bold
# plt.rcParams.update({'font.weight': 'bold'})
# load the loss data from CSV files
loss_data1 = pd.read_csv('./loss_record/loss_sensitivity_analysis_v2_1_1.csv')
loss_data2 = pd.read_csv('./loss_record/loss_sensitivity_analysis_v2_2_1.csv')
loss_data3 = pd.read_csv('./loss_record/loss_sensitivity_analysis_v2_3_1.csv')
loss_data4 = pd.read_csv('./loss_record/loss_sensitivity_analysis_v2_4_1.csv')
loss_data5 = pd.read_csv('./loss_record/loss_sensitivity_analysis_v2_5_1.csv')
# loss_data6 = pd.read_csv('./loss_record/loss_sensitivity_analysis_v2_6_2.csv')
loss_data6 = pd.read_csv('loss_sensitivity_analysis1.csv')
loss_data7 = pd.read_csv('./loss_record/loss_sensitivity_analysis_v2_7_1.csv')
loss_data8 = pd.read_csv('./loss_record/loss_sensitivity_analysis_v2_8_1.csv')
loss_data9 = pd.read_csv('./loss_record/loss_sensitivity_analysis_v2_9_1.csv')
loss_data10 = pd.read_csv('./loss_record/loss_sensitivity_analysis_v2_10_1.csv')

plt.figure(figsize=(10, 6))
plt.plot(loss_data2['Epoch'], loss_data2['label_loss'], label=r'$\mathrm{\lambda_1 = 10, \lambda_2 = 1}$', color='orange', linestyle='--', linewidth=2)
plt.plot(loss_data1['Epoch'], loss_data1['label_loss'], label=r'$\mathrm{\lambda_1 = 1, \lambda_2 = 1}$', color='blue', linestyle='-', linewidth=2)
plt.plot(loss_data6['Epoch'], loss_data6['label_loss'], label=r'$\mathrm{\lambda_1 = 1, \lambda_2 = 0.001}$', color='brown', linestyle='--', linewidth=2)
plt.plot(loss_data10['Epoch'], loss_data10['label_loss'], label=r'$\mathrm{\lambda_1 = 0.5, \lambda_2 = 0.1}$', color='lime', linestyle='-.', linewidth=2)
plt.plot(loss_data9['Epoch'], loss_data9['label_loss'], label=r'$\mathrm{\lambda_1 = 0.5, \lambda_2 = 0.001}$', color='cyan', linestyle='-', linewidth=2)
plt.plot(loss_data7['Epoch'], loss_data7['label_loss'], label=r'$\mathrm{\lambda_1 = 0.2, \lambda_2 = 1}$', color='pink', linestyle='-.', linewidth=2)
plt.plot(loss_data8['Epoch'], loss_data8['label_loss'], label=r'$\mathrm{\lambda_1 = 0.2, \lambda_2 = 0.001}$', color='gray', linestyle=':', linewidth=2)
plt.plot(loss_data3['Epoch'], loss_data3['label_loss'], label=r'$\mathrm{\lambda_1 = 0.1, \lambda_2 = 1}$', color='green', linestyle='-.', linewidth=2)
plt.plot(loss_data4['Epoch'], loss_data4['label_loss'], label=r'$\mathrm{\lambda_1 = 0.1, \lambda_2 = 0.02}$', color='red', linestyle=':', linewidth=2)
plt.plot(loss_data5['Epoch'], loss_data5['label_loss'], label=r'$\mathrm{\lambda_1 = 0.1, \lambda_2 = 0.001}$', color='purple', linestyle='-', linewidth=2)
# legend two columns

plt.yticks(np.arange(1.2, 2.2, 0.3))
plt.legend(ncol=2)
# change the y axis 
# set xlabel bold
plt.xlabel('Epochs')
plt.ylabel('Pattern label loss')
plt.savefig('./loss_record/loss_sensitivity_analysis_ll.pdf', bbox_inches='tight', dpi=300)
plt.show()

# Training losses: duration

In [ ]:
plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'font.family': 'Arial'})
plt.figure(figsize=(10, 6))

plt.plot(loss_data2['Epoch'], loss_data2['duration_loss'], label=r'$\mathrm{\lambda_1 = 10, \lambda_2 = 1}$', color='orange', linestyle='--', linewidth=2)
plt.plot(loss_data1['Epoch'], loss_data1['duration_loss'], label=r'$\mathrm{\lambda_1 = 1, \lambda_2 = 1}$', color='blue', linestyle='-', linewidth=2)
plt.plot(loss_data6['Epoch'], loss_data6['duration_loss']*1000, label=r'$\mathrm{\lambda_1 = 1, \lambda_2 = 0.001}$', color='brown', linestyle='--', linewidth=2)
plt.plot(loss_data10['Epoch'], loss_data10['duration_loss'], label=r'$\mathrm{\lambda_1 = 0.5, \lambda_2 = 0.1}$', color='lime', linestyle='-.', linewidth=2)
plt.plot(loss_data9['Epoch'], loss_data9['duration_loss'], label=r'$\mathrm{\lambda_1 = 0.5, \lambda_2 = 0.001}$', color='cyan', linestyle='-', linewidth=2)
plt.plot(loss_data7['Epoch'], loss_data7['duration_loss'], label=r'$\mathrm{\lambda_1 = 0.2, \lambda_2 = 1}$', color='pink', linestyle='-.', linewidth=2)
plt.plot(loss_data8['Epoch'], loss_data8['duration_loss'], label=r'$\mathrm{\lambda_1 = 0.2, \lambda_2 = 0.001}$', color='gray', linestyle=':', linewidth=2)
plt.plot(loss_data3['Epoch'], loss_data3['duration_loss'], label=r'$\mathrm{\lambda_1 = 0.1, \lambda_2 = 1}$', color='green', linestyle='-.', linewidth=2)
plt.plot(loss_data4['Epoch'], loss_data4['duration_loss'], label=r'$\mathrm{\lambda_1 = 0.1, \lambda_2 = 0.02}$', color='red', linestyle=':', linewidth=2)
plt.plot(loss_data5['Epoch'], loss_data5['duration_loss'], label=r'$\mathrm{\lambda_1 = 0.1, \lambda_2 = 0.001}$', color='purple', linestyle='-', linewidth=2)

plt.legend(ncol=2)
plt.xlabel('Epochs')
plt.ylabel('Pattern duration loss')
plt.savefig('./loss_record/loss_sensitivity_analysis_dl.pdf', bbox_inches='tight', dpi=300)
plt.show()

# Save model

In [ ]:
# Save the model's state dictionary
torch.save(model.state_dict(), 'model2.pth')
print("Model saved successfully")

# Create the model instance
model = PatternDurationPredictor(num_labels=6)

# Load the model's state dictionary
model.load_state_dict(torch.load('model.pth'))
model.eval()  # Set the model to evaluation mode
print("Model loaded successfully")


test: a real sequence is [2, 2, 4, 3, 1, 1],duration: [20, 15, 43, 34, 25, 42]

# Test module

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# Define the model class (should match the original definition)
class PatternDurationPredictor(nn.Module):
    def __init__(self, num_labels, embedding_dim=20, lstm_hidden_dim=25, num_lstm_layers=6, stats_dim=4):
        super(PatternDurationPredictor, self).__init__()
        self.embedding = nn.Embedding(num_labels, embedding_dim)
        self.lstm_input_dim = embedding_dim + num_labels  # Adjust input size
        self.lstm = nn.LSTM(input_size=self.lstm_input_dim, hidden_size=lstm_hidden_dim, num_layers=num_lstm_layers, batch_first=True, dropout=0.5)
        
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.sigmoid = nn.Sigmoid()
        self.attention1 = nn.Linear(lstm_hidden_dim + stats_dim, 1)
        self.attention2 = nn.Linear(lstm_hidden_dim + stats_dim, 1)
        self.fc1 = nn.Linear(lstm_hidden_dim + stats_dim, 10)  # For label prediction
        self.fc2 = nn.Linear(lstm_hidden_dim + stats_dim, 18)  # For duration prediction
        self.fc3 = nn.Linear(18, 10)  # For duration prediction
        self.fc4 = nn.Linear(10, 1)  # For duration prediction
        
        self.fc5 = nn.Linear(10, num_labels)  # For duration prediction

    def forward(self, x, transitions, lengths, stats):
        x = self.embedding(x)
        x = torch.cat((x, transitions), dim=2)  # Concatenate embedding and transition features

        packed_input = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        packed_output, (h_n, _) = self.lstm(packed_input)
        h_n = h_n[-1]  # Take the last hidden state
        
        h_n = torch.cat((h_n, stats), dim=1)

        attention_weights1 = nn.functional.softmax(self.attention1(h_n), dim=1)
        attention_output1 = (h_n * attention_weights1)

        h_n1 = self.sigmoid(attention_output1)
        
        attention_weights2 = nn.functional.softmax(self.attention2(h_n1), dim=1)
        attention_output2 = (h_n1 * attention_weights2)
        h_n2 = self.sigmoid(attention_output2)
        
        label_output = self.fc5(self.fc1(h_n1))
        
        duration_output = self.fc2(h_n2)
        duration_output = self.fc3(duration_output)
        duration_output = self.fc4(duration_output)
        duration_output = self.relu(duration_output)
        
        label_output = self.softmax(label_output)
        
        return label_output, duration_output

# Load the model
model = PatternDurationPredictor(num_labels=6)
model.load_state_dict(torch.load('model.pth'))
model.eval()
print("Model loaded successfully")

# Sample data preparation
# Note: Replace these sample data with actual sample data
sample_labels = [2, 2, 4, 3, 1]  # Example label sequence
sample_transitions = [transition_probs[label] for label in sample_labels]
sample_stats = duration_stats[sample_labels[-1]]
# print(sample_labels, sample_transitions, sample_stats)
# Convert to tensors
sample_labels_tensor = torch.tensor(sample_labels, dtype=torch.long).unsqueeze(0)  # Add batch dimension
sample_transitions_tensor = torch.tensor(sample_transitions, dtype=torch.float).unsqueeze(0)  # Add batch dimension
sample_stats_tensor = torch.tensor(sample_stats, dtype=torch.float).unsqueeze(0)  # Add batch dimension
sample_lengths = [len(sample_labels)]

# Make predictions
with torch.no_grad():
    predicted_label, predicted_duration = model(sample_labels_tensor, sample_transitions_tensor, sample_lengths, sample_stats_tensor)

# Output results
print(f'Predicted label probabilities: {predicted_label.squeeze()}')
print(f'Predicted label: {torch.argmax(predicted_label, dim=1).item()}')
print(f'Predicted duration: {predicted_duration.item()}')
